Simulamos nulos

In [4]:
# =============================================================================
# 01_simular_nulos_corregido.py
#
# Paso 1: Simulación de Datos Faltantes (CORREGIDO para no superponer bloques)
#
# Esta celda genera los datasets con un 20% de valores nulos simulados 
# en diferentes configuraciones, garantizando que los bloques no se solapen.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
import time

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado. Asegúrate de que esté en la carpeta 'CSVs' dentro de tu proyecto.")
    df_air_quality = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv en la ruta esperada.")
    print("Creando DataFrame de ejemplo para continuar con la ejecución del script...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([
                _date, _city_ex,
                200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5,
                50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20,
                40+np.random.rand()*20
            ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
if 'CO2' in df_air_quality.columns:
    df_air_quality = df_air_quality.drop(columns=['CO2'])
    print("Columna 'CO2' descartada.")

# --- Parámetros de Simulación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL
N_REPETITIONS = 5

# --- Escenarios con un 20% de nulos totales ---
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
os.makedirs(SIMULATED_DATA_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Simulación de NaNs (CORREGIDA)---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    np.random.seed(random_seed_input)
    random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True)
    n_rows = len(df_with_nans)
    row_indices_to_nan_positions = []
    scenario_type = scenario_config_input['type']
    config = scenario_config_input['config']
    
    if scenario_type == 'row_percentage':
        percentage = config['percentage']
        num_rows_to_nan = int(n_rows * percentage)
        if num_rows_to_nan > 0 and n_rows > 0:
            row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    
    elif scenario_type == 'block_rows':
        block_length = config['length']
        percentage = config.get('percentage', 0.20)
        total_rows_to_nan = int(n_rows * percentage)
        num_blocks = max(1, total_rows_to_nan // block_length) if block_length > 0 else 0
        
        # --- LÓGICA CORREGIDA PARA EVITAR SUPERPOSICIÓN ---
        used_indices = set()
        possible_starts = list(range(n_rows - block_length + 1))
        random.shuffle(possible_starts)
        
        blocks_created = 0
        for start_pos in possible_starts:
            if blocks_created >= num_blocks:
                break
            
            # Verificar si el nuevo bloque se superpone con los existentes
            end_pos = start_pos + block_length
            block_range = set(range(start_pos, end_pos))
            if not block_range.intersection(used_indices):
                row_indices_to_nan_positions.extend(list(block_range))
                used_indices.update(block_range)
                blocks_created += 1
        
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))
        
    if row_indices_to_nan_positions:
        col_positions_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_to_nan = [pos for pos in col_positions_to_nan if pos != -1]
        if col_positions_to_nan:
             df_with_nans.iloc[row_indices_to_nan_positions, col_positions_to_nan] = np.nan
             
    return df_with_nans

# --- Script Principal de Simulación ---
start_time_script = time.time()

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está cargado o está vacío. Carga tus datos primero.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original} (Directorio seguro: {city_name_safe})")
    city_sim_data_dir_root = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    os.makedirs(city_sim_data_dir_root, exist_ok=True)

    df_city_original_for_sim = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city_original_for_sim.empty:
        print(f"  No hay datos para la ciudad {city_name_original}. Omitiendo.")
        continue
        
    if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Advertencia: NaNs preexistentes en {city_name_original}. Rellenando con ffill/bfill para la simulación.")
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_original_for_sim.columns:
                df_city_original_for_sim[col_fill] = df_city_original_for_sim[col_fill].ffill().bfill()
        if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
            print(f"  Error: No se pudieron rellenar NaNs preexistentes en {city_name_original}. Omitiendo ciudad.")
            continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Generando escenario: {scenario_name_safe}")
        scenario_city_sim_data_dir_specific = os.path.join(city_sim_data_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_sim_data_dir_specific, exist_ok=True)

        for rep in range(1, N_REPETITIONS + 1):
            current_seed = int(time.time() * 1000) + random.randint(0,100000) + rep + hash(city_name_original) + hash(scenario_name_safe)
            current_seed = current_seed % (2**32 - 1)
            
            df_city_with_nans = introduce_nans_by_row(
                df_city_original_for_sim, scenario, random_seed_input=current_seed
            )
            
            sim_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_with_nans.csv"
            df_city_with_nans.to_csv(os.path.join(scenario_city_sim_data_dir_specific, sim_filename), index=False)
            
    print(f"  Simulación de escenarios para {city_name_original} completada.")

end_time_script = time.time()
print(f"\nProceso de simulación de NaNs completado.")
print(f"Archivos de datos simulados guardados en subcarpetas dentro de: {SIMULATED_DATA_ROOT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv
Columna 'CO2' descartada.

Procesando Ciudad: Brasilia (Directorio seguro: Brasilia)
  Generando escenario: RandomRows_20percent
  Generando escenario: BlockRows_12h_20percent
  Generando escenario: BlockRows_24h_20percent
  Generando escenario: BlockRows_36h_20percent
  Generando escenario: BlockRows_48h_20percent
  Generando escenario: BlockRows_60h_20percent
  Simulación de escenarios para Brasilia completada.

Procesando Ciudad: Cairo (Directorio seguro: Cairo)
  Generando escenario: RandomRows_20percent
  Generando escenario: BlockRows_12h_20percent
  Generando escenario: BlockRows_24h_20percent
  Generando escenario: BlockRows_36h_20percent
  Generando escenario: BlockRows_48h_20percent
  Generando escenario: BlockRows_60h_20percent
  Simulación de escenarios para Cairo completada.

Procesando Ciudad: Dubai (Directorio seguro: Dubai)
  Generando escenario: RandomRows_20percent
  Generando escenario: BlockRows_12h_20percent
  Generando es

Interpolación Lineal

In [5]:
# =============================================================================
# 02_imputacion_lineal.py
#
# Paso 2: Imputación con Interpolación Lineal y Evaluación
#
# Esta celda lee los archivos con NaNs generados, les aplica interpolación 
# lineal y evalúa su rendimiento.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import glob

# --- 0. Carga de Datos ORIGINALES (Ground Truth) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_ground_truth = pd.read_csv(file_path_load)
    print(f"Datos 'Ground Truth' cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv. Es necesario para la evaluación.")
    exit()

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])

# --- Parámetros de Imputación y Evaluación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL

# --- Lista de escenarios a procesar (DEBE COINCIDIR CON LA CELDA 1) ---
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_12h_20percent', 'type': 'block_rows', 'config': {'length': 12, 'percentage': 0.20}},
    {'name': 'BlockRows_24h_20percent', 'type': 'block_rows', 'config': {'length': 24, 'percentage': 0.20}},
    {'name': 'BlockRows_36h_20percent', 'type': 'block_rows', 'config': {'length': 36, 'percentage': 0.20}},
    {'name': 'BlockRows_48h_20percent', 'type': 'block_rows', 'config': {'length': 48, 'percentage': 0.20}},
    {'name': 'BlockRows_60h_20percent', 'type': 'block_rows', 'config': {'length': 60, 'percentage': 0.20}}
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
METRICS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'metricas')
IMPUTED_CSVS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'CSVs')

os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares para Métricas ---
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            
            nrmae_range = np.nan
            nrmse_range = np.nan
            if original_col_range > 1e-9:
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range

            results[col]['MAE'] = mae; results[col]['RMSE'] = rmse
            results[col]['NRMAE_range'] = nrmae_range; results[col]['NRMSE_range'] = nrmse_range
        except Exception as e: pass
    return results

# --- Script Principal de Imputación y Evaluación ---
all_detailed_metrics_results = []
start_time_script = time.time()

for city_name_original in df_air_quality_ground_truth['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original}")
    
    df_city_gt = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == city_name_original].copy()
    if df_city_gt[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
            if col_fill in df_city_gt.columns: df_city_gt[col_fill] = df_city_gt[col_fill].ffill().bfill()
    if df_city_gt.isnull().values.any():
        print(f"  No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo ciudad.")
        continue
    df_city_original_ground_truth_reset = df_city_gt.reset_index(drop=True)

    sim_data_city_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    if not os.path.exists(sim_data_city_dir):
        print(f"  Directorio de datos simulados no encontrado para {city_name_original}. Omitiendo.")
        continue

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Evaluando escenario: {scenario_name_safe}")
        
        sim_data_scenario_dir = os.path.join(sim_data_city_dir, scenario_name_safe)
        if not os.path.exists(sim_data_scenario_dir):
            print(f"    Directorio de escenario no encontrado: {sim_data_scenario_dir}. Omitiendo.")
            continue
            
        imputed_csv_scenario_dir = os.path.join(IMPUTED_CSVS_OUTPUT_DIR, city_name_safe, scenario_name_safe)
        os.makedirs(imputed_csv_scenario_dir, exist_ok=True)

        repetition_files = glob.glob(os.path.join(sim_data_scenario_dir, "*_with_nans.csv"))

        for sim_file_path in repetition_files:
            file_name = os.path.basename(sim_file_path)
            try:
                df_city_with_nans_reset = pd.read_csv(sim_file_path)
                rep_str = file_name.split('_rep')[-1].split('_')[0]
                seed_str = file_name.split('_seed')[-1].split('_')[0]
                rep = int(rep_str) if rep_str.isdigit() else 0
                current_seed = int(seed_str) if seed_str.isdigit() else 0
            except Exception as e:
                print(f"      Error cargando {file_name}: {e}")
                continue
            
            # 1. Realizar imputación
            df_city_imputed_linear_reset = df_city_with_nans_reset.copy()
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].interpolate(
                        method='linear', limit_direction='both', limit_area=None)
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].ffill().bfill()

            # 2. Guardar archivo imputado
            imputed_filename = file_name.replace('_with_nans.csv', f'_imputed_{INTERPOLATION_METHOD_NAME}.csv')
            df_city_imputed_linear_reset.to_csv(os.path.join(imputed_csv_scenario_dir, imputed_filename), index=False)

            # 3. Calcular métricas
            nan_mask = df_city_with_nans_reset[ALL_TARGET_COLS].isnull() & ~df_city_original_ground_truth_reset[ALL_TARGET_COLS].isnull()
            metrics_scores = calculate_metrics_with_nmae_nrmse(df_city_original_ground_truth_reset, df_city_imputed_linear_reset, nan_mask)

            # 4. Almacenar resultados detallados
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed, 'Imputation_Method': INTERPOLATION_METHOD_NAME,
                    'Variable': var_name, 
                    'MAE': metrics_scores[var_name]['MAE'], 
                    'RMSE': metrics_scores[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores[var_name]['NRMAE_range'],
                    'NRMSE_range': metrics_scores[var_name]['NRMSE_range']
                })
    print(f"  Procesamiento de escenarios para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_results)
    
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original in grouped_by_city_var['City'].unique():
        city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_original]
        variables_summary_path = os.path.join(METRICS_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_original} guardado en: {variables_summary_path}")
        if city_name_original == df_air_quality_ground_truth['City'].unique()[0]: print(city_metrics_df)

    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
    print(f"  Resumen combinado de contaminantes guardado en: {contaminants_summary_path}")
    print(contaminants_combined_summary)

    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False)
    print(f"  Resumen combinado de AQI guardado en: {aqi_summary_path}")
    print(aqi_combined_summary)
else:
    print("No se generaron resultados de métricas detalladas.")

end_time_script = time.time()
print(f"\nProceso de imputación y evaluación completado.")
print(f"Archivos de datos simulados leídos desde: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos CSV imputados guardados en subcarpetas dentro de: {IMPUTED_CSVS_OUTPUT_DIR}")
print(f"Archivos de métricas resumidas guardados en: {METRICS_OUTPUT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos 'Ground Truth' cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad: Brasilia
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Brasilia completado.

Procesando Ciudad: Cairo
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluando escenario: BlockRows_48h_20percent
  Evaluando escenario: BlockRows_60h_20percent
  Procesamiento de escenarios para Cairo completado.

Procesando Ciudad: Dubai
  Evaluando escenario: RandomRows_20percent
  Evaluando escenario: BlockRows_12h_20percent
  Evaluando escenario: BlockRows_24h_20percent
  Evaluando escenario: BlockRows_36h_20percent
  Evaluan

SARIMA ORDERS

In [ ]:
# CELDA 1: BÚSQUEDA DE ÓRDENES ÓPTIMOS ARIMA/SARIMA (EJECUTAR SOLO UNA VEZ O CUANDO SEA NECESARIO)
import pandas as pd
import numpy as np
import os
import pmdarima as pm
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- 0. Carga de Datos (Asegúrate de que df_air_quality esté cargado) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality = pd.read_csv(file_path_load)
    df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
    if 'CO2' in df_air_quality.columns: df_air_quality = df_air_quality.drop(columns=['CO2'])
    print(f"Datos cargados desde: {file_path_load} para Celda 1")
except FileNotFoundError:
    print("Error: df_air_quality no encontrado en Celda 1.")
    _example_data = [] # Datos de ejemplo mínimos
    _cities_example = ['Brasilia', 'Cairo']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(24*7*4): # 4 semanas de datos horarios
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 40+np.random.rand()*20 ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'AQI'])
    print("DataFrame de ejemplo creado para Celda 1.")


# --- Parámetros ---
COLUMNS_FOR_ARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
SEASONAL_PERIOD = 24

PROJECT_ROOT_DIR_ARIMA_SEARCH = os.getcwd()
# Directorio donde se guardarán los JSONs directamente
ORDERS_OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR_ARIMA_SEARCH, 'imputaciones', 'SARIMA_auto', 'model_orders_v2')
os.makedirs(ORDERS_OUTPUT_DIR, exist_ok=True) # Solo se crea la carpeta base 'model_orders'

print(f"--- INICIO DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")
print(f"Los órdenes se guardarán directamente en: {ORDERS_OUTPUT_DIR}")

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está definido o está vacío en Celda 1.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando ciudad: {city_name_original}")

    df_city = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city.empty:
        print(f"  No hay datos para {city_name_original}, omitiendo.")
        continue

    for col_name in COLUMNS_FOR_ARIMA:
        if col_name not in df_city.columns:
            print(f"  Variable {col_name} no encontrada en los datos de {city_name_original}, omitiendo.")
            continue

        print(f"  Buscando orden para variable: {col_name}...")
        series_data = df_city[col_name].dropna()

        if len(series_data) < 2 * SEASONAL_PERIOD:
            print(f"    Datos insuficientes para SARIMA ({len(series_data)} puntos). Intentando ARIMA no estacional.")
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, d=None, seasonal=False, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)
                order = auto_model.order; seasonal_order = (0,0,0,0)
            except Exception as e:
                print(f"    Error en auto_arima no estacional para {col_name}: {e}. Usando (1,0,0).")
                order = (1,0,0); seasonal_order = (0,0,0,0)
        elif len(series_data) < 10:
             print(f"    Datos muy insuficientes (<10 puntos). Usando (1,0,0).")
             order = (1,0,0); seasonal_order = (0,0,0,0)
        else:
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, m=SEASONAL_PERIOD, start_P=0, D=None, max_P=2, max_Q=2, seasonal=True, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)     
                order = auto_model.order; seasonal_order = auto_model.seasonal_order
                print(f"    Orden encontrado para {col_name}: {order}, Estacional: {seasonal_order}")
            except Exception as e:
                print(f"    Error en auto_arima para {col_name}: {e}. Usando (1,0,1)(1,0,1,{SEASONAL_PERIOD}).")
                order = (1,0,1); seasonal_order = (1,0,1,SEASONAL_PERIOD) if SEASONAL_PERIOD > 0 else (0,0,0,0)
        
        order_data = {'order': list(order), 'seasonal_order': list(seasonal_order)}
        # Guardar directamente en ORDERS_OUTPUT_DIR, sin subcarpeta de ciudad para el JSON
        file_path_order = os.path.join(ORDERS_OUTPUT_DIR, f'orders_{city_name_safe}_{col_name}.json')
        with open(file_path_order, 'w') as f:
            json.dump(order_data, f, indent=4)
        print(f"    Órdenes guardados en: {file_path_order}")

print(f"--- FIN DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
--- Fase 1: Encontrando/Cargando Órdenes SARIMA Óptimos ---
  Procesando ciudad para encontrar órdenes: Brasilia
    Procesando variable para auto_arima: CO
      Ejecutando auto_arima para Brasilia-CO (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-CO: Order=(3, 0, 0), Seasonal=(1, 0, 1, 24)
    Procesando variable para auto_arima: NO2
      Ejecutando auto_arima para Brasilia-NO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-NO2: Order=(1, 1, 0), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: SO2
      Ejecutando auto_arima para Brasilia-SO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-SO2: Order=(3, 1, 1), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: O3


KeyboardInterrupt: 

Imputación SARIMA

In [9]:
# =============================================================================
# 02_imputacion_sarima.py (Versión Adaptada)
#
# Imputación con SARIMA usando órdenes guardados y los nuevos escenarios de NaNs.
# Enfocado en una sola ciudad y una repetición para ejecución rápida.
# =============================================================================

import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
import glob

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning) # Para statsmodels

# --- Parámetros de Imputación y Evaluación ---
TARGET_CITY = "Brasilia" # <--- MODIFICACIÓN: Enfocarse en una sola ciudad
COLUMNS_TO_IMPUTE_SARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_REPETITIONS_SARIMA = 1 # <--- MODIFICACIÓN: Reducido a 1 para rapidez

# --- MODIFICACIÓN: Nuevos escenarios de simulación de NaNs ---
NAN_SIMULATION_SCENARIOS_SARIMA = [
    {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_12h_20percent'},
    {'name': 'BlockRows_24h_20percent'},
    {'name': 'BlockRows_36h_20percent'},
    {'name': 'BlockRows_48h_20percent'},
    {'name': 'BlockRows_60h_20percent'}
]

# --- Rutas de Entrada y Salida ---
PROJECT_ROOT_DIR_SARIMA_IMPUTE = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'CSVs', 'simulados')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', 'SARIMA', 'model_orders')
SARIMA_METHOD_NAME = 'SARIMA'
IMPUTATIONS_SARIMA_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', SARIMA_METHOD_NAME)
METRICS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'metricas')
IMPUTED_CSVS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'CSVs')
os.makedirs(METRICS_SARIMA_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, exist_ok=True)

# --- Función de Imputación SARIMA MODIFICADA ---
def impute_sarima_kalman_smoother(series_with_nans, order_tuple, seasonal_order_tuple, original_series_for_bounds=None):
    imputed_series = series_with_nans.copy()
    original_nan_mask = imputed_series.isnull()
    
    if not original_nan_mask.any():
        return series_with_nans

    min_len_heuristic = max(sum(order_tuple), sum(seasonal_order_tuple[:3])) + seasonal_order_tuple[3] if seasonal_order_tuple[3] > 0 else max(sum(order_tuple), sum(seasonal_order_tuple[:3]))
    min_len_heuristic = max(min_len_heuristic, 10)

    if len(imputed_series.dropna()) < min_len_heuristic + 5:
        return series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()

    sarima_failed_or_unstable = False
    try:
        model = SARIMAX(imputed_series, 
                        order=order_tuple,
                        seasonal_order=seasonal_order_tuple,
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                        initialization='approximate_diffuse')
        results = model.fit(disp=False, maxiter=200)
        predictions_all = results.predict(start=imputed_series.index[0], end=imputed_series.index[-1])
        imputed_series.loc[original_nan_mask] = predictions_all[original_nan_mask]

        if imputed_series.isnull().any():
            temp_linear_fill = series_with_nans.interpolate(method='linear', limit_direction='both')
            imputed_series = imputed_series.fillna(temp_linear_fill).ffill().bfill()

    except Exception as e:
        print(f"      Error en SARIMA: {e}. Usando fallback a lineal.")
        sarima_failed_or_unstable = True

    if sarima_failed_or_unstable:
        imputed_series = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    
    if original_series_for_bounds is not None and original_nan_mask.any():
        clean_original_data = original_series_for_bounds.dropna()
        if not clean_original_data.empty:
            q01 = clean_original_data.quantile(0.001)
            q99 = clean_original_data.quantile(0.999)
            iqr_val = clean_original_data.quantile(0.75) - clean_original_data.quantile(0.25)
            padding = max(1.0, 0.1 * abs(q99 - q01)) if (q99 - q01) != 0 else 1.0
            lower_bound = q01 - 3 * iqr_val - padding
            upper_bound = q99 + 3 * iqr_val + padding
            
            extreme_values_mask = original_nan_mask & ((imputed_series < lower_bound) | (imputed_series > upper_bound))
            if extreme_values_mask.any():
                temp_linear_imputation = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
                imputed_series.loc[extreme_values_mask] = temp_linear_imputation[extreme_values_mask]
    
    return imputed_series

# --- Función para calcular métricas ---
def calculate_metrics_sarima(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    mae_vals, rmse_vals = {}, {}
    for col in COLUMNS_TO_IMPUTE_SARIMA:
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan; continue
        
        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values
        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]
        
        if len(true_at_nans) == 0:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan; continue
        
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]
        
        if len(true_at_nans_valid) == 0:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan; continue
        
        try:
            mae_vals[col] = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse_vals[col] = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
        except Exception:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan
    return mae_vals, rmse_vals

# --- Script Principal de Imputación SARIMA ---
all_detailed_metrics_sarima_results = []
start_time_script_sarima = time.time()
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig):
            file_path_load_orig = alt_file_path_load_orig
        else:
            raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_original_gt = pd.read_csv(file_path_load_orig)
    df_air_quality_original_gt['Date'] = pd.to_datetime(df_air_quality_original_gt['Date'])
    if 'CO2' in df_air_quality_original_gt.columns:
        df_air_quality_original_gt = df_air_quality_original_gt.drop(columns=['CO2'])
    print(f"Datos originales GT cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error crítico: Archivo 'Air_Quality.csv' de ground truth no encontrado.")
    exit()

city_name_original_sarima = TARGET_CITY
city_name_safe_sarima = city_name_original_sarima.replace(" ", "_").replace("-", "_")
print(f"\nProcesando Ciudad para Imputación SARIMA: {city_name_original_sarima}")

city_sim_data_input_dir_root = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR, city_name_safe_sarima)
city_imputed_csv_sarima_dir_root = os.path.join(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, city_name_safe_sarima)
os.makedirs(city_imputed_csv_sarima_dir_root, exist_ok=True)

df_city_gt_for_metrics = df_air_quality_original_gt[df_air_quality_original_gt['City'] == city_name_original_sarima].copy()
if df_city_gt_for_metrics.empty:
    print(f"No hay datos para {city_name_original_sarima}. Saliendo.")
    exit()

if df_city_gt_for_metrics[COLUMNS_TO_IMPUTE_SARIMA].isnull().any().any():
    for col_fill_gt in COLUMNS_TO_IMPUTE_SARIMA:
        if col_fill_gt in df_city_gt_for_metrics.columns:
            df_city_gt_for_metrics[col_fill_gt] = df_city_gt_for_metrics[col_fill_gt].ffill().bfill()
df_city_gt_for_metrics_reset = df_city_gt_for_metrics.reset_index(drop=True)

for scenario_sarima in NAN_SIMULATION_SCENARIOS_SARIMA:
    scenario_name_safe_sarima = scenario_sarima['name'].replace("%", "percent").replace(" ", "_")
    print(f"  Escenario: {scenario_name_safe_sarima}")
    scenario_city_sim_data_input_dir = os.path.join(city_sim_data_input_dir_root, scenario_name_safe_sarima)
    scenario_city_imputed_csv_sarima_dir = os.path.join(city_imputed_csv_sarima_dir_root, scenario_name_safe_sarima)
    os.makedirs(scenario_city_imputed_csv_sarima_dir, exist_ok=True)

    for rep_s in range(1, N_REPETITIONS_SARIMA + 1):
        sim_files_in_dir = sorted([f for f in os.listdir(scenario_city_sim_data_input_dir) if f.startswith(f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir:
            print(f"    Advertencia: No se encontró archivo simulado para rep {rep_s}. Omitiendo.")
            continue
        
        sim_file_path_to_load = os.path.join(scenario_city_sim_data_input_dir, sim_files_in_dir[0])
        current_seed_loaded = sim_files_in_dir[0].split('_seed')[-1].split('_with_nans.csv')[0]
        
        try:
            df_city_with_nans_loaded = pd.read_csv(sim_file_path_to_load)
        except FileNotFoundError:
            print(f"    Error: Archivo simulado no encontrado: {sim_file_path_to_load}"); continue
        
        df_city_imputed_sarima = df_city_with_nans_loaded.copy()
        mask_for_metrics_s = pd.DataFrame(False, index=df_city_gt_for_metrics_reset.index, columns=COLUMNS_TO_IMPUTE_SARIMA)
        for col_mask in COLUMNS_TO_IMPUTE_SARIMA:
             if col_mask in df_city_with_nans_loaded.columns and col_mask in df_city_gt_for_metrics_reset.columns:
                mask_for_metrics_s[col_mask] = df_city_with_nans_loaded[col_mask].isnull() & ~df_city_gt_for_metrics_reset[col_mask].isnull()

        for col_impute_s in COLUMNS_TO_IMPUTE_SARIMA:
            print(f"    Imputando columna: {col_impute_s}...")
            order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe_sarima}_{col_impute_s}.json')
            try:
                with open(order_file_path, 'r') as f: orders = json.load(f)
                order_tuple = tuple(orders['order']); seasonal_order_tuple = tuple(orders['seasonal_order'])
            except (FileNotFoundError, json.JSONDecodeError) as e:
                print(f"      Archivo de órdenes no encontrado o corrupto: {order_file_path}. Usando (1,0,1)(0,0,0,0).")
                order_tuple = (1,0,1); seasonal_order_tuple = (0,0,0,0)

            series_to_impute = df_city_with_nans_loaded[col_impute_s].copy()
            original_clean_series_for_bounds = df_city_gt_for_metrics_reset[col_impute_s].copy()

            imputed_values_col = impute_sarima_kalman_smoother(
                series_to_impute, order_tuple, seasonal_order_tuple,
                original_series_for_bounds=original_clean_series_for_bounds
            )
            df_city_imputed_sarima[col_impute_s] = imputed_values_col
        
        imputed_sarima_filename = f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_seed{current_seed_loaded}_imputed_sarima.csv"
        df_city_imputed_sarima.to_csv(os.path.join(scenario_city_imputed_csv_sarima_dir, imputed_sarima_filename), index=False)
        
        mae_s, rmse_s = calculate_metrics_sarima(df_city_gt_for_metrics_reset, df_city_imputed_sarima, mask_for_metrics_s)
        for var_n in COLUMNS_TO_IMPUTE_SARIMA:
            all_detailed_metrics_sarima_results.append({
                'City': city_name_original_sarima, 'Scenario': scenario_name_safe_sarima, 'Repetition': rep_s,
                'Seed': current_seed_loaded, 'Imputation_Method': SARIMA_METHOD_NAME, 'Variable': var_n,
                'MAE': mae_s.get(var_n, np.nan), 'RMSE': rmse_s.get(var_n, np.nan)
            })

# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_sarima_results:
    df_all_detailed_s = pd.DataFrame(all_detailed_metrics_sarima_results)
    
    # Resumen por variable y escenario para la ciudad
    city_scenario_summary = df_all_detailed_s.groupby(['City', 'Scenario', 'Variable'])[['MAE', 'RMSE']].mean().reset_index()
    city_metrics_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'metrics_summary_{SARIMA_METHOD_NAME}_{city_name_safe_sarima}.csv')
    city_scenario_summary.to_csv(city_metrics_path, index=False)
    print(f"\nResumen SARIMA para {city_name_original_sarima} guardado en: {city_metrics_path}")
    print(city_scenario_summary)
    
    # Archivo detallado
    global_detailed_s_file = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'ALL_metrics_detailed_{SARIMA_METHOD_NAME}_{city_name_safe_sarima}.csv')
    df_all_detailed_s.to_csv(global_detailed_s_file, index=False)
    print(f"\nMétricas detalladas SARIMA para {city_name_original_sarima} guardadas en: {global_detailed_s_file}")

end_time_script_sarima = time.time()
print(f"\nProceso SARIMA completado. CSVs: {IMPUTED_CSVS_SARIMA_OUTPUT_DIR}, Métricas: {METRICS_SARIMA_OUTPUT_DIR}")
print(f"Tiempo SARIMA: {(end_time_script_sarima - start_time_script_sarima) / 60:.2f} min.")

Datos originales GT cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación SARIMA: Brasilia
  Escenario: RandomRows_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_12h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_24h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O3...
    Imputando columna: PM2.5...
    Imputando columna: PM10...
    Imputando columna: AQI...
  Escenario: BlockRows_36h_20percent
    Imputando columna: CO...
    Imputando columna: NO2...
    Imputando columna: SO2...
    Imputando columna: O

RANDOM FOREST

In [4]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler # Para escalar features
from sklearn.pipeline import Pipeline # Para combinar scaler e imputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_orig_load = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_orig_load = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_orig_load['Date'] = pd.to_datetime(df_air_quality_orig_load['Date'])
if 'CO2' in df_air_quality_orig_load.columns:
    df_air_quality_orig_load = df_air_quality_orig_load.drop(columns=['CO2'])

# --- Parámetros ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL # Columnas que originalmente tendrán NaNs y serán evaluadas

# Features de tiempo
df_air_quality_orig_load['Hour'] = df_air_quality_orig_load['Date'].dt.hour
df_air_quality_orig_load['DayOfWeek'] = df_air_quality_orig_load['Date'].dt.dayofweek
df_air_quality_orig_load['Month'] = df_air_quality_orig_load['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# Creación de Lags (ANTES de la simulación de NaNs en el bucle principal)
N_LAGS = [1, 2, 24, 48] # Lags a crear para cada variable en ALL_TARGET_COLS
LAG_FEATURES = []
df_with_features = df_air_quality_orig_load.copy()
for col_to_lag in ALL_TARGET_COLS:
    for lag_val in N_LAGS:
        lag_col_name = f"{col_to_lag}_lag{lag_val}"
        df_with_features[lag_col_name] = df_with_features.groupby('City')[col_to_lag].shift(lag_val)
        LAG_FEATURES.append(lag_col_name)

# Columnas que el imputer usará como features. Incluye originales, de tiempo y lags.
# El imputer intentará imputar NaNs en CUALQUIERA de estas si los encuentra.
COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM = ALL_TARGET_COLS + TIME_FEATURES + LAG_FEATURES

N_REPETITIONS = 1
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados', 'Brasilia') # Leer desde aquí

RF_IMPUTER_METHOD_NAME = 'RandomForestImputer' # Nombre descriptivo
IMPUTATIONS_RF_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', RF_IMPUTER_METHOD_NAME)
METRICS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'metricas')
IMPUTED_CSVS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'CSVs')

os.makedirs(METRICS_RF_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_RF_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares ---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    # df_original_city_input ya viene con lags y features de tiempo
    np.random.seed(random_seed_input)
    random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True) # Trabajar con copia e índice reseteado
    n_rows = len(df_with_nans)
    row_indices_to_nan_positions = [] # Posiciones (0 a n-1)

    scenario_type = scenario_config_input['type']
    config = scenario_config_input['config']

    if scenario_type == 'row_percentage':
        percentage = config['percentage']
        num_rows_to_nan = int(n_rows * percentage)
        if num_rows_to_nan > 0 and n_rows > 0:
            row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length = config['length']
        num_blocks_ratio = config.get('num_blocks_ratio', 0.01)
        num_blocks = max(1, int((n_rows * num_blocks_ratio) / block_length if block_length > 0 else n_rows * num_blocks_ratio))
        for _ in range(num_blocks):
            if n_rows < block_length or (n_rows - block_length) < 0: break
            start_pos = random.randint(0, n_rows - block_length)
            row_indices_to_nan_positions.extend(list(range(start_pos, start_pos + block_length)))
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))

    if row_indices_to_nan_positions:
        # Poner NaN SOLO en las columnas objetivo originales (ALL_TARGET_COLS)
        # Las columnas de lag que se vuelvan NaN como consecuencia de esto está bien.
        # Las features de tiempo NO deben tener NaNs introducidos.
        col_positions_target_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_target_to_nan = [pos for pos in col_positions_target_to_nan if pos != -1]
        if col_positions_target_to_nan:
            df_with_nans.iloc[row_indices_to_nan_positions, col_positions_target_to_nan] = np.nan
    
    # Máscara de NaNs introducidos (solo para ALL_TARGET_COLS)
    df_original_city_reset = df_original_city_input.copy().reset_index(drop=True)
    introduced_nan_mask_df = pd.DataFrame(False, index=df_with_nans.index, columns=ALL_TARGET_COLS)
    for col_mask in ALL_TARGET_COLS:
        if col_mask in df_with_nans.columns and col_mask in df_original_city_reset.columns:
             introduced_nan_mask_df[col_mask] = df_with_nans[col_mask].isnull() & ~df_original_city_reset[col_mask].isnull()
    
    return df_with_nans, introduced_nan_mask_df


def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    # y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset solo contienen ALL_TARGET_COLS
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue
        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values
        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            # Usar el rango del ground truth completo de la columna para la normalización
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0

            nrmae_range, nrmse_range = np.nan, np.nan
            if original_col_range > 1e-9:
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range
            results[col]['MAE'], results[col]['RMSE'] = mae, rmse
            results[col]['NRMAE_range'], results[col]['NRMSE_range'] = nrmae_range, nrmse_range
        except Exception: pass
    return results

# --- Script Principal ---
all_detailed_metrics_rf_results = []
script_start_time_rf = time.time()

if 'df_with_features' not in locals() or df_with_features.empty:
    print("Error: df_with_features (datos con lags y tiempo) no está definido o está vacío.")
    exit()

for city_name_original in df_with_features['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación RF: {city_name_original}")

    # Datos originales (con lags y features de tiempo) para esta ciudad, usados como base para simular NaNs
    df_city_original_features = df_with_features[df_with_features['City'] == city_name_original].copy()
    if df_city_original_features.empty: continue

    # Ground truth (sin lags, solo ALL_TARGET_COLS) para calcular métricas, pre-rellenado
    df_city_gt_eval = df_air_quality_orig_load[df_air_quality_orig_load['City'] == city_name_original].copy()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_gt_eval.columns:
                df_city_gt_eval[col_fill] = df_city_gt_eval[col_fill].ffill().bfill()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any(): # Si aún hay NaNs, omitir
        print(f"  Error fatal: No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo.")
        continue
    df_city_original_ground_truth_reset = df_city_gt_eval.reset_index(drop=True) # Solo columnas originales + tiempo


    city_imputed_csv_rf_dir_root = os.path.join(IMPUTED_CSVS_RF_OUTPUT_DIR, city_name_safe)
    os.makedirs(city_imputed_csv_rf_dir_root, exist_ok=True)

    # Definir el Imputer y el Scaler
    # --- AJUSTES EN PARÁMETROS DEL IMPUTER (VERSIÓN RÁPIDA PARA PRUEBA) ---
    rf_estimator_test = RandomForestRegressor(
        n_estimators=15,       # MUY BAJO, solo para prueba de velocidad
        random_state=42,
        n_jobs=-1,             # Usar todos los cores
        max_depth=7,           # Reducido drásticamente
        min_samples_split=10,  # Razonable
        min_samples_leaf=5,    # Razonable
        # max_features='sqrt'  # Puedes mantenerlo o quitarlo para prueba
    )
    iterative_imputer_rf = IterativeImputer(
        estimator=rf_estimator_test,
        max_iter=8,              # MUY BAJO, solo para ver si converge algo
        tol=1e-2,                # Tolerancia menos estricta
        random_state=42,
        initial_strategy='mean', # 'mean' es a menudo más rápido que 'median' para el inicio
        # imputation_order='ascending', # El default, puede ser más rápido que 'random'
        verbose=1                # PONER verbose=1 PARA VER QUÉ ESTÁ HACIENDO
    )
    scaler = StandardScaler() # Usaremos scaler antes del imputer

    # Ajustar el Scaler y el Imputer usando los datos ORIGINALES COMPLETOS de la ciudad
    # (con features de tiempo y lags, pero SIN los NaNs simulados artificialmente aún)
    # Los NaNs de los lags (al principio de la serie) serán manejados por el imputer.
    data_for_fitting_imputer = df_city_original_features[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
    
    print(f"  Ajustando Scaler e IterativeImputer con RandomForest para {city_name_original}...")
    start_fit_time = time.time()
    try:
        # 1. Escalar los datos
        scaled_data_for_fitting = scaler.fit_transform(data_for_fitting_imputer)
        # 2. Ajustar el imputer con los datos escalados
        iterative_imputer_rf.fit(scaled_data_for_fitting)
        fit_duration = time.time() - start_fit_time
        print(f"  Scaler e Imputer ajustados para {city_name_original} en {fit_duration:.2f} segundos.")
    except Exception as e_fit:
        print(f"  Error ajustando Scaler/IterativeImputer para {city_name_original}: {e_fit}. Omitiendo ciudad.")
        continue


    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"    Escenario: {scenario_name_safe}")
        
        # Los datos simulados con NaNs se LEEN desde las carpetas generadas por el script anterior
        # No se vuelven a generar aquí.
        # sim_data_scenario_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe, scenario_name_safe)
        sim_data_scenario_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, scenario_name_safe) # CAMBIAR, SOLO ES PARA PRUEBA
        imputed_rf_scenario_dir = os.path.join(city_imputed_csv_rf_dir_root, scenario_name_safe)
        os.makedirs(imputed_rf_scenario_dir, exist_ok=True)

        if not os.path.exists(sim_data_scenario_dir):
            print(f"      Directorio de datos simulados no encontrado: {sim_data_scenario_dir}. Omitiendo escenario.")
            continue

        for rep in range(1, N_REPETITIONS + 1):
            matching_files = [f for f in os.listdir(sim_data_scenario_dir) if f.startswith(f"{city_name_safe}_{scenario_name_safe}_rep{rep}") and f.endswith("_with_nans.csv")]
            if not matching_files: continue
            sim_file_path_to_load = os.path.join(sim_data_scenario_dir, matching_files[0])
            current_seed_loaded = matching_files[0].split('_seed')[-1].split('_with_nans.csv')[0]

            try:
                # df_city_with_nans_loaded_raw contendrá Date, City, y ALL_TARGET_COLS con NaNs simulados
                df_city_with_nans_loaded_raw = pd.read_csv(sim_file_path_to_load)
                
                # Reconstruir df_city_with_nans_features para el imputer:
                # Empezar con el df original de la ciudad (que tiene lags y time features)
                df_for_imputation_base = df_city_original_features.copy().reset_index(drop=True)
                
                # Sobrescribir las columnas ALL_TARGET_COLS con los valores (y NaNs) del archivo simulado
                # Asegurarse de que los índices estén alineados (ambos reseteados)
                df_city_with_nans_loaded_reset = df_city_with_nans_loaded_raw.reset_index(drop=True)
                for col_target in ALL_TARGET_COLS:
                    if col_target in df_city_with_nans_loaded_reset.columns:
                        df_for_imputation_base[col_target] = df_city_with_nans_loaded_reset[col_target]
                
                # Ahora df_for_imputation_base tiene:
                # - Lags (con NaNs al principio y donde los originales se volvieron NaN)
                # - Time features (completas)
                # - ALL_TARGET_COLS (con NaNs simulados)

            except Exception as e_load:
                print(f"      Error cargando/procesando {sim_file_path_to_load}: {e_load}.")
                continue
            
            # La máscara de NaNs introducidos se refiere a las columnas ALL_TARGET_COLS
            # Se basa en la comparación con el ground truth original SIN lags.
            temp_introduced_nan_mask_df = pd.DataFrame(False, index=df_city_original_ground_truth_reset.index, columns=ALL_TARGET_COLS)
            for col_mask_check in ALL_TARGET_COLS:
                 if col_mask_check in df_city_with_nans_loaded_reset.columns and col_mask_check in df_city_original_ground_truth_reset.columns:
                    temp_introduced_nan_mask_df[col_mask_check] = df_city_with_nans_loaded_reset[col_mask_check].isnull() & \
                                                                ~df_city_original_ground_truth_reset[col_mask_check].isnull()

            # Preparar datos para el imputer (solo columnas relevantes)
            df_to_impute_for_rf = df_for_imputation_base[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
            
            try:
                start_transform_time = time.time()
                # 1. Escalar los datos con NaNs
                scaled_data_to_impute = scaler.transform(df_to_impute_for_rf)
                # 2. Imputar los datos escalados
                imputed_scaled_data_rf_array = iterative_imputer_rf.transform(scaled_data_to_impute)
                # 3. Desescalar los datos imputados
                imputed_data_rf_array = scaler.inverse_transform(imputed_scaled_data_rf_array)
                transform_duration = time.time() - start_transform_time
                
                # Reconstruir el DataFrame imputado
                df_imputed_features = pd.DataFrame(imputed_data_rf_array, columns=COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM)
                
                # Crear el DataFrame final para guardar (Date, City, ALL_TARGET_COLS imputados, y TIME_FEATURES)
                df_final_imputed_rf = df_for_imputation_base[['Date', 'City']].copy() # Mantener Date y City originales
                for col_final in ALL_TARGET_COLS + TIME_FEATURES: # Guardar también time features
                    if col_final in df_imputed_features.columns:
                         df_final_imputed_rf[col_final] = df_imputed_features[col_final]

            except Exception as e_impute:
                print(f"      Error durante la transformación/imputación RF para rep {rep}: {e_impute}. Omitiendo.")
                continue

            imputed_filename_rf = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed_loaded}_imputed_rf.csv"
            df_final_imputed_rf.to_csv(os.path.join(imputed_rf_scenario_dir, imputed_filename_rf), index=False)

            metrics_scores_rf = calculate_metrics_with_nmae_nrmse(
                df_city_original_ground_truth_reset[ALL_TARGET_COLS], # Ground truth (solo cols objetivo)
                df_final_imputed_rf[ALL_TARGET_COLS],                 # Imputados (solo cols objetivo)
                temp_introduced_nan_mask_df                           # Máscara (solo cols objetivo)
            )
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_rf_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed_loaded, 'Imputation_Method': RF_IMPUTER_METHOD_NAME,
                    'Variable': var_name, 'MAE': metrics_scores_rf[var_name]['MAE'], 
                    'RMSE': metrics_scores_rf[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores_rf[var_name]['NRMAE_range'],
                    'NRMSE_range': metrics_scores_rf[var_name]['NRMSE_range']
                })
    print(f"  Procesamiento de escenarios RF para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas (sin cambios en esta sección) ---
if all_detailed_metrics_rf_results:
    df_all_detailed_metrics_rf = pd.DataFrame(all_detailed_metrics_rf_results)
    print("\nGenerando resúmenes de métricas RF por variable y ciudad...")
    grouped_by_city_var_rf = df_all_detailed_metrics_rf.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var_rf.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original_summary in grouped_by_city_var_rf['City'].unique(): # Renombrar variable de bucle
        city_name_safe_rf_summary = city_name_original_summary.replace(" ", "_").replace("-", "_")
        city_metrics_df_rf = grouped_by_city_var_rf[grouped_by_city_var_rf['City'] == city_name_original_summary]
        variables_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_rf_summary}.csv')
        city_metrics_df_rf.to_csv(variables_summary_path_rf, index=False)
        print(f"  Resumen RF de variables para {city_name_original_summary} guardado en: {variables_summary_path_rf}")
        if city_name_original_summary == df_air_quality_orig_load['City'].unique()[0]: print(city_metrics_df_rf)

    print("\nGenerando resumen RF combinado de contaminantes...")
    df_contaminants_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary_rf = df_contaminants_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary_rf.to_csv(contaminants_summary_path_rf, index=False)
    print(f"  Resumen RF combinado de contaminantes guardado en: {contaminants_summary_path_rf}")
    print(contaminants_combined_summary_rf)

    print("\nGenerando resumen RF combinado de AQI...")
    df_aqi_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'] == 'AQI']
    aqi_combined_summary_rf = df_aqi_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary_rf.to_csv(aqi_summary_path_rf, index=False)
    print(f"  Resumen RF combinado de AQI guardado en: {aqi_summary_path_rf}")
    print(aqi_combined_summary_rf)
else:
    print("No se generaron resultados de métricas detalladas para Random Forest Imputer.")

script_end_time_rf = time.time()
print(f"\nProceso de Imputación con Random Forest completado.")
print(f"Archivos de datos simulados leídos desde: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos de métricas resumidas RF guardados en: {METRICS_RF_OUTPUT_DIR}")
print(f"Archivos CSV imputados con RF guardados en subcarpetas dentro de: {IMPUTED_CSVS_RF_OUTPUT_DIR}")
print(f"Tiempo total de ejecución RF: {(script_end_time_rf - script_start_time_rf) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación RF: Brasilia
  Ajustando Scaler e IterativeImputer con RandomForest para Brasilia...
[IterativeImputer] Completing matrix with shape (8784, 38)
[IterativeImputer] Change: 4.597718620147441, scaled tolerance: 0.08004094305323933 
[IterativeImputer] Change: 2.125996006557909, scaled tolerance: 0.08004094305323933 
[IterativeImputer] Change: 1.0292146182090784, scaled tolerance: 0.08004094305323933 
[IterativeImputer] Change: 1.0224159326807918, scaled tolerance: 0.08004094305323933 
[IterativeImputer] Change: 0.6215913142017488, scaled tolerance: 0.08004094305323933 
[IterativeImputer] Change: 0.47286816227548456, scaled tolerance: 0.08004094305323933 
[IterativeImputer] Change: 0.5285062924040936, scaled tolerance: 0.08004094305323933 
[IterativeImputer] Change: 0.591723113915037, scaled tolerance: 0.08004094305323933 
  Scaler e Imputer ajustados para Brasilia en 55.32 segundos.
    Escenario: RandomRows_10pe

LSTM-VAE

In [8]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
import keras # Keras 3 de alto nivel
from keras import layers, Model, optimizers
from keras import backend as K
import time
import glob
import warnings
import shutil
import re
from unicodedata import normalize
import joblib
import random # Asegurar que random esté importado

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing TF Names ---
def sanitize_tf_name(name_str):
    s = normalize('NFKD', name_str); s = s.encode('ascii', 'ignore'); s = s.decode('ascii')
    s = re.sub(r'[^a-zA-Z0-9_]', '_', s)
    if s and s[0].isdigit() and not s.isdigit(): s = "n_" + s
    if not s: s = "sanitized_default_name"
    return s

# --- 0. Carga de Datos Originales ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_full_original = pd.read_csv(file_path_load)
    print(f"Datos originales cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200): # Reducido para ejemplo rápido
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_full_original = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_full_original['Date'] = pd.to_datetime(df_air_quality_full_original['Date'])
if 'CO2' in df_air_quality_full_original.columns:
    df_air_quality_full_original = df_air_quality_full_original.drop(columns=['CO2'])
df_air_quality_full_original['Hour'] = df_air_quality_full_original['Date'].dt.hour
df_air_quality_full_original['DayOfWeek'] = df_air_quality_full_original['Date'].dt.dayofweek
df_air_quality_full_original['Month'] = df_air_quality_full_original['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# --- Parámetros ---
TARGET_CITY_VAE = "Brasilia"
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS_VAE = CONTAMINANT_COLS + AQI_COL # Columnas a imputar individualmente

N_LAGS_VAE = 24 * 1 
LATENT_DIM_VAE = 8 
ENCODER_NEURONS_L1_VAE = 32 
DECODER_NEURONS_L1_VAE = 32 
VAE_INITIAL_LR = 1e-3; KL_LOSS_WEIGHT = 0.0001 
VAE_BATCH_SIZE = 64 
WFV_INITIAL_TRAIN_POINTS_MIN_RATIO = 0.1 
WFV_VALIDATION_POINTS_MIN_RATIO = 0.05
WFV_STEP_POINTS_MIN_RATIO = 0.05
WFV_EPOCHS_PER_FOLD = 50 # Mantener bajo para pruebas
WFV_ES_PATIENCE = 15  
WFV_LR_PATIENCE = 8      
MAX_WFV_FOLDS = 1 # Reducido para pruebas rápidas
N_REPETITIONS_FOR_EVAL = 1 # Para la evaluación de cada modelo VAE por variable
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
# MODIFICACIÓN: Nombre de la carpeta de salida para reflejar que son CSVs unificados
LSTM_VAE_METHOD_NAME = 'LSTM_VAE_imputer'
IMPUTATIONS_LSTM_VAE_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', LSTM_VAE_METHOD_NAME)
METRICS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'metricas')
IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'CSVs') # Aquí irán los CSVs UNIFICADOS
MODELS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'model_weights')
os.makedirs(METRICS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)

# --- Funciones VAE (sin cambios lógicos, solo cómo se usan sus resultados) ---
# ... (sampling, VAE class, create_vae_model_architecture, create_sequences_for_vae_training, impute_segment_with_lstm_vae) ...
# (COPIAR Y PEGAR LAS DEFINICIONES DE ESTAS FUNCIONES DE TU CELDA ANTERIOR DE IMPUTACIÓN VAE)
def sampling(args):
    z_mean, z_log_var = args; batch = tf.shape(z_mean)[0]; dim = tf.shape(z_mean)[1]
    epsilon = tf.random.normal(shape=(batch, dim), mean=0., stddev=1.0)
    return z_mean + keras.ops.exp(0.5 * z_log_var) * epsilon
class VAE(keras.Model):
    def __init__(self, encoder, decoder, kl_loss_weight_param, name="vae_model", **kwargs):
        super().__init__(name=name, **kwargs)
        self.encoder_model = encoder; self.decoder_model = decoder
        self.kl_loss_weight = kl_loss_weight_param
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
        self.mse_loss_fn = keras.losses.MeanSquaredError(reduction="sum_over_batch_size")
    @property
    def metrics(self): return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]
    def call(self, inputs, training=False):
        z_mean, z_log_var, z = self.encoder_model(inputs, training=training)
        return self.decoder_model(z, training=training)
    def train_step(self, data):
        x_input = data[0] if isinstance(data, tuple) else data
        target_for_loss = x_input[:, :, 0:1]
        with tf.GradientTape() as tape:
            z_mean_val, z_log_var_val, z_val = self.encoder_model(x_input, training=True)
            reconstruction = self.decoder_model(z_val, training=True)
            reconstruction_loss_val = self.mse_loss_fn(target_for_loss, reconstruction)
            kl_loss_val = -0.5 * keras.ops.mean(keras.ops.sum(1 + z_log_var_val - keras.ops.square(z_mean_val) - keras.ops.exp(z_log_var_val), axis=1))
            total_loss_val = reconstruction_loss_val + self.kl_loss_weight * kl_loss_val
        grads = tape.gradient(total_loss_val, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss_val); self.reconstruction_loss_tracker.update_state(reconstruction_loss_val); self.kl_loss_tracker.update_state(kl_loss_val)
        return {m.name: m.result() for m in self.metrics}
    def test_step(self, data):
        x_input = data[0] if isinstance(data, tuple) else data
        target_for_loss = x_input[:, :, 0:1]
        z_mean_val, z_log_var_val, z_val = self.encoder_model(x_input, training=False)
        reconstruction = self.decoder_model(z_val, training=False)
        reconstruction_loss_val = self.mse_loss_fn(target_for_loss, reconstruction)
        kl_loss_val = -0.5 * keras.ops.mean(keras.ops.sum(1 + z_log_var_val - keras.ops.square(z_mean_val) - keras.ops.exp(z_log_var_val), axis=1))
        total_loss_val = reconstruction_loss_val + self.kl_loss_weight * kl_loss_val
        self.total_loss_tracker.update_state(total_loss_val); self.reconstruction_loss_tracker.update_state(reconstruction_loss_val); self.kl_loss_tracker.update_state(kl_loss_val)
        return {m.name: m.result() for m in self.metrics}
def create_vae_model_architecture(n_lags, n_features_input, latent_dim, enc_neurons, dec_neurons, var_name_for_layer_names="var"):
    safe_var_prefix = sanitize_tf_name(var_name_for_layer_names)
    enc_in = layers.Input(shape=(n_lags, n_features_input), name=f"{safe_var_prefix}_encoder_input")
    enc_lstm = layers.LSTM(enc_neurons, activation='tanh', name=f"{safe_var_prefix}_encoder_lstm")(enc_in)
    z_m = layers.Dense(latent_dim, name=f"{safe_var_prefix}_z_mean")(enc_lstm)
    z_lv = layers.Dense(latent_dim, name=f"{safe_var_prefix}_z_log_var")(enc_lstm)
    z = layers.Lambda(sampling, name=f"{safe_var_prefix}_z_sampling_layer")([z_m, z_lv])
    encoder = Model(enc_in, [z_m, z_lv, z], name=f"{safe_var_prefix}_encoder")
    dec_in = layers.Input(shape=(latent_dim,), name=f"{safe_var_prefix}_decoder_latent_input")
    dec_rep = layers.RepeatVector(n_lags, name=f"{safe_var_prefix}_decoder_repeat_vector")(dec_in)
    dec_lstm = layers.LSTM(dec_neurons, activation='tanh', return_sequences=True, name=f"{safe_var_prefix}_decoder_lstm")(dec_rep)
    dec_out = layers.TimeDistributed(layers.Dense(1), name=f"{safe_var_prefix}_decoder_reconstruction_output")(dec_lstm)
    decoder = Model(dec_in, dec_out, name=f"{safe_var_prefix}_decoder")
    return encoder, decoder
def create_sequences_for_vae_training(target_series_1d_scaled, exog_df_scaled_nullable, n_lags_vae):
    X_seq_list = []
    num_target_features = 1
    num_exog_features = exog_df_scaled_nullable.shape[1] if exog_df_scaled_nullable is not None and not exog_df_scaled_nullable.empty else 0
    for i in range(len(target_series_1d_scaled) - n_lags_vae + 1):
        target_window_reshaped = target_series_1d_scaled[i : i + n_lags_vae].reshape(-1, num_target_features)
        if num_exog_features > 0:
            exog_window = exog_df_scaled_nullable.iloc[i : i + n_lags_vae].values
            if exog_window.shape[0] != n_lags_vae: continue
            combined_window = np.concatenate((target_window_reshaped, exog_window), axis=1)
        else: combined_window = target_window_reshaped
        X_seq_list.append(combined_window)
    if not X_seq_list: return np.array([]).reshape(0, n_lags_vae, num_target_features + num_exog_features)
    return np.array(X_seq_list)
def impute_segment_with_lstm_vae(target_series_1d_with_nans, exog_df_for_segment_scaled, vae_model, scaler_target, n_lags, n_total_features_vae_input, fallback_value_unscaled):
    imputed_series_unscaled = target_series_1d_with_nans.copy().flatten()
    nan_indices_in_segment = np.where(np.isnan(imputed_series_unscaled))[0]
    num_target_features, num_exog_features_expected = 1, n_total_features_vae_input - 1
    for idx_nan in nan_indices_in_segment:
        if idx_nan < n_lags:
            imputed_series_unscaled[idx_nan] = imputed_series_unscaled[idx_nan-1] if idx_nan > 0 and not np.isnan(imputed_series_unscaled[idx_nan-1]) else fallback_value_unscaled
            continue
        current_target_window_unscaled = imputed_series_unscaled[idx_nan - n_lags : idx_nan].copy()
        s_temp_target = pd.Series(current_target_window_unscaled); s_temp_target = s_temp_target.ffill().bfill()
        if s_temp_target.isnull().any(): s_temp_target.fillna(fallback_value_unscaled, inplace=True)
        current_target_window_scaled = scaler_target.transform(s_temp_target.values.reshape(-1, 1))
        current_exog_window_scaled = np.array([]).reshape(n_lags,0)
        if num_exog_features_expected > 0 and exog_df_for_segment_scaled is not None:
            start_exog_window, end_exog_window = idx_nan - n_lags, idx_nan
            if start_exog_window >=0 and end_exog_window <= len(exog_df_for_segment_scaled): current_exog_window_scaled = exog_df_for_segment_scaled.iloc[start_exog_window:end_exog_window].values
            else: current_exog_window_scaled = np.zeros((n_lags, num_exog_features_expected))
        elif num_exog_features_expected > 0: current_exog_window_scaled = np.zeros((n_lags, num_exog_features_expected))
        vae_input_sequence_scaled = np.concatenate((current_target_window_scaled, current_exog_window_scaled), axis=1) if num_exog_features_expected > 0 else current_target_window_scaled
        if vae_input_sequence_scaled.shape != (n_lags, n_total_features_vae_input):
            imputed_series_unscaled[idx_nan] = imputed_series_unscaled[idx_nan-1] if idx_nan > 0 and not np.isnan(imputed_series_unscaled[idx_nan-1]) else fallback_value_unscaled
            continue
        vae_input_reshaped = vae_input_sequence_scaled.reshape(1, n_lags, n_total_features_vae_input)
        try:
            reconstructed_scaled_sequence = vae_model.predict(vae_input_reshaped, verbose=0)
            imputed_value_scaled = reconstructed_scaled_sequence[0, -1, 0]
            if np.isnan(imputed_value_scaled) or np.isinf(imputed_value_scaled): raise ValueError("Pred VAE NaN/Inf")
            imputed_series_unscaled[idx_nan] = scaler_target.inverse_transform(np.array([[imputed_value_scaled]]))[0,0]
        except Exception: imputed_series_unscaled[idx_nan] = imputed_series_unscaled[idx_nan-1] if idx_nan > 0 and not np.isnan(imputed_series_unscaled[idx_nan-1]) else fallback_value_unscaled
    return imputed_series_unscaled.reshape(-1,1)
def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range, nrmse_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan), (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal de Imputación con LSTM-VAE ---
all_detailed_metrics_lstm_vae_results = []
script_start_time_lstm_vae = time.time()
df_city_original_for_wfv_unfiltered = df_air_quality_full_original[df_air_quality_full_original['City'] == TARGET_CITY_VAE].copy()
if df_city_original_for_wfv_unfiltered.empty: print(f"No hay datos para {TARGET_CITY_VAE}. Saliendo."); exit()

trained_vaes = {}
scalers_target = {}
# scaler_exog_city se entrena una vez para la ciudad
df_exog_base_city = df_city_original_for_wfv_unfiltered[TIME_FEATURES].copy()
for tf_col in TIME_FEATURES: df_exog_base_city[tf_col] = df_exog_base_city[tf_col].ffill().bfill()
if df_exog_base_city.isnull().any().any(): print(f"Error: NaNs en TFs globales para {TARGET_CITY_VAE}."); exit()
scaler_exog_city = MinMaxScaler(feature_range=(0,1))
scaled_exog_global_city_df = pd.DataFrame(scaler_exog_city.fit_transform(df_exog_base_city), index=df_exog_base_city.index, columns=df_exog_base_city.columns)
city_model_weights_root_dir = os.path.join(MODELS_LSTM_VAE_OUTPUT_DIR, TARGET_CITY_VAE) # Carpeta para todos los modelos de esta ciudad
os.makedirs(city_model_weights_root_dir, exist_ok=True)
joblib.dump(scaler_exog_city, os.path.join(city_model_weights_root_dir, f'scaler_exog_GLOBAL_{TARGET_CITY_VAE}.joblib'))
n_exog_features_global = scaled_exog_global_city_df.shape[1]
n_features_input_global_vae = 1 + n_exog_features_global

# --- Bucle de Entrenamiento de Modelos VAE (uno por variable) ---
for target_var_to_impute in ALL_TARGET_COLS_VAE:
    K.clear_session()
    base_seed_val = int(time.time_ns() / 1000) + hash(target_var_to_impute + TARGET_CITY_VAE)
    tf.random.set_seed(abs(base_seed_val + os.getpid() + 1) % (2**32 - 1))
    np.random.seed(abs(base_seed_val + os.getpid() + 2) % (2**32 - 1))
    random.seed(abs(base_seed_val + os.getpid() + 3) % (2**32 - 1))

    print(f"\n--- Preparando VAE para VARIABLE: {target_var_to_impute} (Ciudad: {TARGET_CITY_VAE}) ---")
    series_clean_target_wfv = df_city_original_for_wfv_unfiltered[target_var_to_impute].ffill().bfill()
    if series_clean_target_wfv.isnull().any(): print(f"  Serie {target_var_to_impute} NaN. Omitiendo."); continue
    
    current_scaler_target_var = MinMaxScaler(feature_range=(0,1))
    scaled_target_wfv_full_for_var = current_scaler_target_var.fit_transform(series_clean_target_wfv.values.reshape(-1,1))
    scalers_target[target_var_to_impute] = current_scaler_target_var

    var_model_weights_dir = os.path.join(city_model_weights_root_dir, sanitize_tf_name(target_var_to_impute)) # Subcarpeta por variable
    os.makedirs(var_model_weights_dir, exist_ok=True)
    joblib.dump(current_scaler_target_var, os.path.join(var_model_weights_dir, f'scaler_target_{sanitize_tf_name(target_var_to_impute)}.joblib'))
    
    print(f"  Iniciando WFV para VAE de {target_var_to_impute}...")
    total_clean_points_var = len(series_clean_target_wfv)
    wfv_init_train_len = int(total_clean_points_var * WFV_INITIAL_TRAIN_POINTS_MIN_RATIO)
    wfv_val_len = int(total_clean_points_var * WFV_VALIDATION_POINTS_MIN_RATIO)
    wfv_step_len = max(1, int(total_clean_points_var * WFV_STEP_POINTS_MIN_RATIO))

    if total_clean_points_var < wfv_init_train_len + wfv_val_len + N_LAGS_VAE + 10:
        print(f"  Datos insuficientes para WFV de {target_var_to_impute}. Omitiendo entrenamiento.");
        trained_vaes[target_var_to_impute] = None; continue

    current_train_end_idx_wfv, fold_num_wfv, best_val_loss_overall_wfv = wfv_init_train_len, 0, float('inf')
    vae_model_weights_path_final_var = os.path.join(var_model_weights_dir, f'vae_weights_final_{sanitize_tf_name(target_var_to_impute)}.weights.h5')
    fold_best_weights_path_temp_var = os.path.join(var_model_weights_dir, f'temp_fold_best_{sanitize_tf_name(target_var_to_impute)}.weights.h5')
    
    current_encoder_arch_vae, current_decoder_arch_vae = create_vae_model_architecture(N_LAGS_VAE, n_features_input_global_vae, LATENT_DIM_VAE, ENCODER_NEURONS_L1_VAE, DECODER_NEURONS_L1_VAE, var_name_for_layer_names=target_var_to_impute)
    
    if os.path.exists(vae_model_weights_path_final_var):
        print(f"  Cargando VAE pre-entrenado para {target_var_to_impute}...")
        final_model_to_load = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT, name=f"vae_loaded_{sanitize_tf_name(target_var_to_impute)}")
        final_model_to_load.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR))
        if not final_model_to_load.built: final_model_to_load.build(input_shape=(None, N_LAGS_VAE, n_features_input_global_vae))
        try:
            final_model_to_load.load_weights(vae_model_weights_path_final_var)
            trained_vaes[target_var_to_impute] = final_model_to_load
            print(f"  Modelo VAE para {target_var_to_impute} cargado."); continue
        except Exception as e_loadw_f: print(f"  Error cargando VAE final para {target_var_to_impute}: {e_loadw_f}. Re-entrenando.")

    while (current_train_end_idx_wfv + wfv_val_len <= total_clean_points_var) and (fold_num_wfv < MAX_WFV_FOLDS):
        fold_num_wfv += 1
        train_target_scaled_fold = scaled_target_wfv_full_for_var[:current_train_end_idx_wfv]
        val_target_scaled_fold = scaled_target_wfv_full_for_var[current_train_end_idx_wfv : current_train_end_idx_wfv + wfv_val_len]
        train_exog_scaled_fold_df = scaled_exog_global_city_df.iloc[:current_train_end_idx_wfv]
        val_exog_scaled_fold_df = scaled_exog_global_city_df.iloc[current_train_end_idx_wfv : current_train_end_idx_wfv + wfv_val_len]
        print(f"    WFV Fold {fold_num_wfv} ({target_var_to_impute}): Train {len(train_target_scaled_fold)}, Val {len(val_target_scaled_fold)}")
        X_train_fold_vae = create_sequences_for_vae_training(train_target_scaled_fold.flatten(), train_exog_scaled_fold_df, N_LAGS_VAE)
        X_val_fold_vae = create_sequences_for_vae_training(val_target_scaled_fold.flatten(), val_exog_scaled_fold_df, N_LAGS_VAE)
        if X_train_fold_vae.shape[0] < VAE_BATCH_SIZE or X_val_fold_vae.shape[0] == 0:
            print(f"      Fold {fold_num_wfv} insuficientes secuencias. Avanzando."); current_train_end_idx_wfv += wfv_step_len; continue
        
        fold_vae_model = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT, name=f"vae_fold{fold_num_wfv}_{sanitize_tf_name(target_var_to_impute)}")
        fold_vae_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR, weight_decay=0.004, clipnorm=1.0))
        if not fold_vae_model.built: fold_vae_model.build(input_shape=(None, N_LAGS_VAE, n_features_input_global_vae))
        if fold_num_wfv > 1 and os.path.exists(vae_model_weights_path_final_var):
            try: fold_vae_model.load_weights(vae_model_weights_path_final_var)
            except Exception as e_lw: print(f"      Error cargando pesos globales en fold: {e_lw}")
        
        callbacks_list = [
            keras.callbacks.EarlyStopping(monitor='val_total_loss', mode='min', patience=WFV_ES_PATIENCE, verbose=0, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_total_loss', mode='min', factor=0.5, patience=WFV_LR_PATIENCE, min_lr=1e-7, verbose=0),
            keras.callbacks.ModelCheckpoint(filepath=fold_best_weights_path_temp_var, monitor='val_total_loss', mode='min', save_best_only=True, save_weights_only=True, verbose=0)
        ]
        fold_history_vae = fold_vae_model.fit(X_train_fold_vae, X_train_fold_vae, epochs=WFV_EPOCHS_PER_FOLD, batch_size=VAE_BATCH_SIZE, validation_data=(X_val_fold_vae, X_val_fold_vae), callbacks=callbacks_list, verbose=0)
        
        current_fold_val_loss = min(fold_history_vae.history.get('val_total_loss', [float('inf')]))
        print(f"      Fold {fold_num_wfv} Val Total Loss: {current_fold_val_loss:.4f}")
        if current_fold_val_loss < best_val_loss_overall_wfv:
            best_val_loss_overall_wfv = current_fold_val_loss
            print(f"      NUEVO MEJOR VAE GLOBAL ({target_var_to_impute}, loss: {best_val_loss_overall_wfv:.4f}). Guardando pesos.")
            # Si ModelCheckpoint guardó, copiar. Sino, guardar el modelo actual (ES lo restauró).
            if os.path.exists(fold_best_weights_path_temp_var) and early_stopping_cb_fold_vae.best == current_fold_val_loss : # Check si el ckpt es el mejor
                 shutil.copyfile(fold_best_weights_path_temp_var, vae_model_weights_path_final_var)
            else: # ES restauró el mejor, o el ckpt no es el mejor de la historia (raro con save_best_only=True y ES restore_best_weights=True)
                fold_vae_model.save_weights(vae_model_weights_path_final_var) # Guardar el modelo actual que ES restauró como el mejor

        if os.path.exists(fold_best_weights_path_temp_var): os.remove(fold_best_weights_path_temp_var)
        current_train_end_idx_wfv += wfv_step_len; K.clear_session()

    final_vae_model_for_var = VAE(current_encoder_arch_vae, current_decoder_arch_vae, KL_LOSS_WEIGHT, name=f"vae_final_{sanitize_tf_name(target_var_to_impute)}")
    final_vae_model_for_var.compile(optimizer=keras.optimizers.AdamW(learning_rate=VAE_INITIAL_LR))
    if not final_vae_model_for_var.built: final_vae_model_for_var.build(input_shape=(None, N_LAGS_VAE, n_features_input_global_vae))
    if os.path.exists(vae_model_weights_path_final_var):
        try: final_vae_model_for_var.load_weights(vae_model_weights_path_final_var); print(f"  Modelo VAE final para {target_var_to_impute} cargado para imputación.")
        except Exception as e_lfw_final: print(f"  Error cargando pesos finales VAE {target_var_to_impute}: {e_lfw_final}."); final_vae_model_for_var = None
    else: print(f"  Pesos VAE no encontrados para {target_var_to_impute}."); final_vae_model_for_var = None
    trained_vaes[target_var_to_impute] = final_vae_model_for_var


# --- Bucle de Imputación de Escenarios y Guardado Unificado ---
print(f"\n--- Iniciando Imputación y Guardado Unificado para {TARGET_CITY_VAE} ---")
df_city_original_gt_reset_for_metrics = df_air_quality_full_original[df_air_quality_full_original['City'] == TARGET_CITY_VAE].reset_index(drop=True)

for scenario_vae in NAN_SIMULATION_SCENARIOS:
    scenario_name_safe_vae = scenario_vae['name'].replace("%", "percent").replace(" ", "_")
    print(f"  Procesando Escenario: {scenario_name_safe_vae}")
    
    # Directorio para los CSVs imputados UNIFICADOS de este escenario
    imputed_unified_scenario_dir = os.path.join(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, TARGET_CITY_VAE.replace(" ","_"), scenario_name_safe_vae)
    os.makedirs(imputed_unified_scenario_dir, exist_ok=True)

    for rep_vae in range(1, N_REPETITIONS_FOR_EVAL + 1):
        sim_data_scenario_dir_for_rep = os.path.join(SIMULATED_DATA_ROOT_DIR, TARGET_CITY_VAE.replace(" ","_"), scenario_name_safe_vae)
        matching_files_sim = [f for f in os.listdir(sim_data_scenario_dir_for_rep) if f.startswith(f"{TARGET_CITY_VAE.replace(' ','_')}_{scenario_name_safe_vae}_rep{rep_vae}") and f.endswith("_with_nans.csv")]
        if not matching_files_sim: print(f"    Archivo _with_nans.csv no encontrado para rep {rep_vae}. Omitiendo."); continue
        sim_file_path_to_load = os.path.join(sim_data_scenario_dir_for_rep, matching_files_sim[0])
        current_seed_loaded_vae = matching_files_sim[0].split('_seed')[-1].split('_with_nans.csv')[0]
        
        try:
            df_with_nans_all_vars_loaded = pd.read_csv(sim_file_path_to_load)
        except Exception as e_load_rep: print(f"    Error cargando {sim_file_path_to_load}: {e_load_rep}."); continue

        df_to_save_imputed_all_vars_rep = df_with_nans_all_vars_loaded.copy()
        exog_for_this_rep_df = pd.DataFrame(index=range(len(df_with_nans_all_vars_loaded)))
        current_scaled_exog_for_rep_df = None
        if n_exog_features_global > 0 and scaler_exog_city is not None:
            temp_dates_rep = pd.to_datetime(df_with_nans_all_vars_loaded['Date'])
            for tf_col_rep in TIME_FEATURES:
                 if tf_col_rep == 'Hour': exog_for_this_rep_df[tf_col_rep] = temp_dates_rep.dt.hour
                 elif tf_col_rep == 'DayOfWeek': exog_for_this_rep_df[tf_col_rep] = temp_dates_rep.dt.dayofweek
                 elif tf_col_rep == 'Month': exog_for_this_rep_df[tf_col_rep] = temp_dates_rep.dt.month
            current_scaled_exog_for_rep_df = pd.DataFrame(scaler_exog_city.transform(exog_for_this_rep_df[TIME_FEATURES]), columns=TIME_FEATURES)
        
        nan_mask_for_metrics_rep = pd.DataFrame(False, index=df_city_original_gt_reset_for_metrics.index, columns=ALL_TARGET_COLS_VAE)
        for col_mask_rep in ALL_TARGET_COLS_VAE:
            if col_mask_rep in df_with_nans_all_vars_loaded.columns and col_mask_rep in df_city_original_gt_reset_for_metrics.columns:
                nan_mask_for_metrics_rep[col_mask_rep] = df_with_nans_all_vars_loaded[col_mask_rep].isnull() & \
                                                       ~df_city_original_gt_reset_for_metrics[col_mask_rep].isnull()

        for target_var_impute_loop in ALL_TARGET_COLS_VAE:
            vae_model_for_var = trained_vaes.get(target_var_impute_loop)
            scaler_target_for_var = scalers_target.get(target_var_impute_loop)
            if vae_model_for_var is None or scaler_target_for_var is None:
                print(f"      VAE o scaler no listo para {target_var_impute_loop}. Usando ffill/bfill para esta variable.")
                df_to_save_imputed_all_vars_rep[target_var_impute_loop] = df_to_save_imputed_all_vars_rep[target_var_impute_loop].ffill().bfill()
                if df_to_save_imputed_all_vars_rep[target_var_impute_loop].isnull().any():
                    df_to_save_imputed_all_vars_rep[target_var_impute_loop].fillna(df_city_original_gt_reset_for_metrics[target_var_impute_loop].mean(), inplace=True)
                continue
            target_series_with_nans_for_impute = df_with_nans_all_vars_loaded[target_var_impute_loop].values
            fallback_value_for_var = df_city_original_gt_reset_for_metrics[target_var_impute_loop].mean()
            imputed_values_for_var_2d = impute_segment_with_lstm_vae(
                target_series_with_nans_for_impute, current_scaled_exog_for_rep_df,
                vae_model_for_var, scaler_target_for_var, N_LAGS_VAE,
                n_features_input_global_vae, fallback_value_unscaled=fallback_value_for_var
            )
            df_to_save_imputed_all_vars_rep[target_var_impute_loop] = imputed_values_for_var_2d.flatten()

        for tf_col_final in TIME_FEATURES:
            if tf_col_final not in df_to_save_imputed_all_vars_rep.columns and tf_col_final in df_city_original_gt_reset_for_metrics.columns:
                 df_to_save_imputed_all_vars_rep[tf_col_final] = df_city_original_gt_reset_for_metrics[tf_col_final].values[:len(df_to_save_imputed_all_vars_rep)]
        
        unified_imputed_filename = f"{TARGET_CITY_VAE.replace(' ','_')}_{scenario_name_safe_vae}_rep{rep_vae}_seed{current_seed_loaded_vae}_imputed_vae_ALL.csv"
        unified_imputed_file_save_path = os.path.join(imputed_unified_scenario_dir, unified_imputed_filename)
        cols_to_save_final = ['Date', 'City'] + ALL_TARGET_COLS_VAE + [tf for tf in TIME_FEATURES if tf in df_to_save_imputed_all_vars_rep.columns]
        cols_to_save_final = [c for c in cols_to_save_final if c in df_to_save_imputed_all_vars_rep.columns]
        df_to_save_imputed_all_vars_rep = df_to_save_imputed_all_vars_rep[cols_to_save_final]
        df_to_save_imputed_all_vars_rep.to_csv(unified_imputed_file_save_path, index=False)
        print(f"      Archivo unificado imputado guardado: {unified_imputed_filename}")
            
        metrics_scores_vae_rep = calculate_metrics_with_nmae_nrmse(
            df_city_original_gt_reset_for_metrics, df_to_save_imputed_all_vars_rep,
            nan_mask_for_metrics_rep, ALL_TARGET_COLS_VAE # Pasar ALL_TARGET_COLS_VAE a la función de métricas
        )
        for var_name_metric in ALL_TARGET_COLS_VAE:
            all_detailed_metrics_lstm_vae_results.append({
                'City': TARGET_CITY_VAE, 'Scenario': scenario_name_safe_vae, 'Repetition': rep_vae, 
                'Seed': current_seed_loaded_vae, 'Imputation_Method': LSTM_VAE_METHOD_NAME, 
                'Variable': var_name_metric, 'MAE': metrics_scores_vae_rep[var_name_metric]['MAE'], 
                'RMSE': metrics_scores_vae_rep[var_name_metric]['RMSE'], 
                'NRMAE_range': metrics_scores_vae_rep[var_name_metric]['NRMAE_range'], 
                'NRMSE_range': metrics_scores_vae_rep[var_name_metric]['NRMSE_range']
            })
        print(f"    Métricas calculadas para rep {rep_vae}, Escenario {scenario_name_safe_vae}")
            
    print(f"  Procesamiento de escenarios para {TARGET_CITY_VAE} completado.")

# --- Guardado de Métricas Resumidas (sin cambios en esta sección) ---
# ... (COPIAR Y PEGAR LA SECCIÓN DE GUARDADO DE MÉTRICAS DE TU SCRIPT ANTERIOR)
if all_detailed_metrics_lstm_vae_results:
    df_all_detailed_metrics_vae = pd.DataFrame(all_detailed_metrics_lstm_vae_results)
    print("\nGenerando resúmenes de métricas LSTM-VAE...")
    detailed_metrics_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'ALL_metrics_detailed_{LSTM_VAE_METHOD_NAME}_{TARGET_CITY_VAE.replace(" ","_")}.csv')
    df_all_detailed_metrics_vae.to_csv(detailed_metrics_path_vae, index=False)
    print(f"  Todas las métricas detalladas LSTM-VAE para {TARGET_CITY_VAE} guardadas en: {detailed_metrics_path_vae}")
    if not df_all_detailed_metrics_vae.empty: print(df_all_detailed_metrics_vae.head())
    grouped_by_var_vae = df_all_detailed_metrics_vae.groupby(['Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_var_vae.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    variables_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_variables_summary_{LSTM_VAE_METHOD_NAME}_{TARGET_CITY_VAE.replace(" ","_")}.csv')
    grouped_by_var_vae.to_csv(variables_summary_path_vae, index=False)
    print(f"  Resumen LSTM-VAE de variables para {TARGET_CITY_VAE} guardado en: {variables_summary_path_vae}")
    print(grouped_by_var_vae)
    df_contaminants_only_vae = df_all_detailed_metrics_vae[df_all_detailed_metrics_vae['Variable'].isin(CONTAMINANT_COLS)]
    if not df_contaminants_only_vae.empty:
        contaminants_combined_summary_vae = df_contaminants_only_vae[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().to_frame().T
        contaminants_combined_summary_vae.insert(0, 'City', TARGET_CITY_VAE); contaminants_combined_summary_vae.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
        contaminants_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{LSTM_VAE_METHOD_NAME}_{TARGET_CITY_VAE.replace(" ","_")}.csv')
        contaminants_combined_summary_vae.to_csv(contaminants_summary_path_vae, index=False); print(f"  Resumen LSTM-VAE contaminantes para {TARGET_CITY_VAE} guardado: {contaminants_summary_path_vae}"); print(contaminants_combined_summary_vae)
    df_aqi_only_vae = df_all_detailed_metrics_vae[df_all_detailed_metrics_vae['Variable'] == 'AQI']
    if not df_aqi_only_vae.empty:
        aqi_combined_summary_vae = df_aqi_only_vae[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().to_frame().T
        aqi_combined_summary_vae.insert(0, 'City', TARGET_CITY_VAE); aqi_combined_summary_vae.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
        aqi_summary_path_vae = os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'metrics_aqi_combined_summary_{LSTM_VAE_METHOD_NAME}_{TARGET_CITY_VAE.replace(" ","_")}.csv')
        aqi_combined_summary_vae.to_csv(aqi_summary_path_vae, index=False); print(f"  Resumen LSTM-VAE AQI para {TARGET_CITY_VAE} guardado: {aqi_summary_path_vae}"); print(aqi_combined_summary_vae)
else: print("No se generaron métricas detalladas para LSTM-VAE.")


script_end_time_lstm_vae = time.time()
print(f"\nProceso LSTM-VAE para {TARGET_CITY_VAE} completado. Tiempo: {(script_end_time_lstm_vae - script_start_time_lstm_vae) / 60:.2f} min.")

Datos originales cargados desde: CSVs\Air_Quality.csv

--- Preparando VAE para VARIABLE: CO (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de CO...
  Cargando VAE pre-entrenado para CO...
  Modelo VAE para CO cargado.

--- Preparando VAE para VARIABLE: NO2 (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de NO2...
  Cargando VAE pre-entrenado para NO2...
  Modelo VAE para NO2 cargado.

--- Preparando VAE para VARIABLE: SO2 (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de SO2...
  Cargando VAE pre-entrenado para SO2...
  Modelo VAE para SO2 cargado.

--- Preparando VAE para VARIABLE: O3 (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de O3...
  Cargando VAE pre-entrenado para O3...
  Modelo VAE para O3 cargado.

--- Preparando VAE para VARIABLE: PM2.5 (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de PM2.5...
  Cargando VAE pre-entrenado para PM2.5...
  Modelo VAE para PM2.5 cargado.

--- Preparando VAE para VARIABLE: PM10 (Ciudad: Brasilia) ---
  Iniciando WFV para VAE de PM10...
  

MOMENT-PRE

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import joblib
import traceback
import time
import random
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos Originales (Ground Truth) ---
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia'] 
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        num_rows_example = 550 
        for i in range(num_rows_example): 
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])
df_air_quality_ground_truth['Hour'] = df_air_quality_ground_truth['Date'].dt.hour
df_air_quality_ground_truth['DayOfWeek'] = df_air_quality_ground_truth['Date'].dt.dayofweek
df_air_quality_ground_truth['Month'] = df_air_quality_ground_truth['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# --- Parámetros Configurables ---
TARGET_CITY_MOMENT = "Brasilia"
CONTAMINANT_COLS_MOMENT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_MOMENT = ['AQI']
ALL_TARGET_COLS_MOMENT = CONTAMINANT_COLS_MOMENT + AQI_COL_MOMENT

SEQ_LEN_MODEL_MOMENT = 128
IMPUTATION_STEP_MOMENT = SEQ_LEN_MODEL_MOMENT
BATCH_SIZE_IMPUTE_MOMENT = 16
MODEL_NAME_MOMENT_PRETRAINED = "AutonLab/MOMENT-1-large"

N_REPETITIONS_MOMENT_EVAL = 5
NAN_SIMULATION_SCENARIOS_MOMENT = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

PROJECT_ROOT_DIR_MOMENT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME = 'MOMENT_PRE' # Nuevo nombre para esta prueba
IMPUTATIONS_MOMENT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_MOMENT, 'imputaciones', MOMENT_METHOD_NAME)
METRICS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'metricas')
IMPUTED_CSVS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'CSVs')
SCALERS_MOMENT_OUTPUT_DIR = os.path.join(IMPUTATIONS_MOMENT_BASE_DIR, 'scalers')
os.makedirs(METRICS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALERS_MOMENT_OUTPUT_DIR, exist_ok=True)

# ======================================================================
# --- Dataset Personalizado para MOMENT (CON LA CORRECCIÓN FINAL) ---
# ======================================================================
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d_multivar_scaled_zeros, nan_mask_original_bool_2d, seq_len, step):
        self.data_array_scaled_zeros = data_array_2d_multivar_scaled_zeros
        self.nan_mask_original = nan_mask_original_bool_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array_scaled_zeros) < self.seq_len : self.n_samples, self.indices = 1, [0]
        elif len(self.data_array_scaled_zeros) == 0: self.n_samples, self.indices = 0, []
        else:
            self.indices = list(range(0, len(self.data_array_scaled_zeros) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array_scaled_zeros) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start :
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array_scaled_zeros) >= self.seq_len: self.indices, self.n_samples = [0],1
            
    def __len__(self): return self.n_samples
    
    def __getitem__(self, idx):
        actual_start_index = self.indices[idx]
        seq_data_scaled_zeros_orig_shape = self.data_array_scaled_zeros[actual_start_index : actual_start_index + self.seq_len]
        seq_input_mask_for_moment_bool_orig_shape = ~self.nan_mask_original[actual_start_index : actual_start_index + self.seq_len]
        
        if len(seq_data_scaled_zeros_orig_shape) < self.seq_len:
            pad_width = self.seq_len - len(seq_data_scaled_zeros_orig_shape)
            seq_data_scaled_zeros_orig_shape = np.pad(seq_data_scaled_zeros_orig_shape, ((0, pad_width), (0, 0)), 'constant', constant_values=0.0)
            seq_input_mask_for_moment_bool_orig_shape = np.pad(seq_input_mask_for_moment_bool_orig_shape, ((0, pad_width), (0,0)), 'constant', constant_values=False)
            
        # El tensor de la serie temporal necesita la forma (n_channels, seq_len)
        timeseries_out = torch.tensor(seq_data_scaled_zeros_orig_shape, dtype=torch.float32).transpose(0, 1)

        # La máscara de entrada NO debe tener la dimensión de canal.
        # Su forma debe ser (seq_len), el DataLoader añadirá la dimensión de batch.
        # Usamos .squeeze() para eliminar la dimensión del canal (que es 1).
        input_mask_out = torch.tensor(seq_input_mask_for_moment_bool_orig_shape, dtype=torch.bool).squeeze()
        
        return {
            'timeseries': timeseries_out,
            'input_mask': input_mask_out
        }

# (El resto del script no necesita cambios)

# --- Función de Imputación con MOMENT ---
def impute_multivar_with_moment_pipeline(df_with_nans_city_scaled_zeros, original_nan_mask_df_bool, moment_pipeline_obj, device_str, scaler_dict_vars_city, seq_len_moment, imputation_step_moment, batch_size_moment):
    imputation_dataset = MomentImputationDataset(df_with_nans_city_scaled_zeros.values, original_nan_mask_df_bool.values, seq_len_moment, imputation_step_moment)
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size_moment, shuffle=False, num_workers=0)
    n_rows_total, n_features_total = df_with_nans_city_scaled_zeros.shape
    accumulated_preds_scaled = np.zeros((n_rows_total, n_features_total), dtype=float)
    accumulated_counts = np.zeros((n_rows_total, n_features_total), dtype=float)
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries_batch_tensor = batch_data['timeseries'].to(device_str)
            input_mask_batch_tensor = batch_data['input_mask'].to(device_str)
            preds_batch_scaled_np = None
            try:
                outputs_obj = moment_pipeline_obj(x_enc=timeseries_batch_tensor, input_mask=input_mask_batch_tensor)
                preds_batch_scaled_np = outputs_obj.reconstruction.cpu().numpy()
            except Exception as e:
                continue
            if preds_batch_scaled_np is None: continue
            for i_in_batch_loop in range(preds_batch_scaled_np.shape[0]):
                current_dataset_idx = batch_idx * batch_size_moment + i_in_batch_loop
                if current_dataset_idx >= len(imputation_loader.dataset.indices): continue
                start_idx_in_segment_loop = imputation_loader.dataset.indices[current_dataset_idx]
                end_idx_in_segment_loop = min(start_idx_in_segment_loop + seq_len_moment, n_rows_total)
                len_window_actual = end_idx_in_segment_loop - start_idx_in_segment_loop
                if len_window_actual > 0:
                    pred_window_scaled_loop = preds_batch_scaled_np[i_in_batch_loop, :, :len_window_actual].T
                    accumulated_preds_scaled[start_idx_in_segment_loop:end_idx_in_segment_loop, :] += pred_window_scaled_loop
                    accumulated_counts[start_idx_in_segment_loop:end_idx_in_segment_loop, :] += 1
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_full_segment_scaled = accumulated_preds_scaled / accumulated_counts
    imputed_full_segment_unscaled_df = pd.DataFrame(columns=df_with_nans_city_scaled_zeros.columns, index=df_with_nans_city_scaled_zeros.index, dtype=float)
    for col_name_unscale in df_with_nans_city_scaled_zeros.columns:
        scaler_var_unscale = scaler_dict_vars_city.get(col_name_unscale)
        if scaler_var_unscale:
            col_idx_unscale = df_with_nans_city_scaled_zeros.columns.get_loc(col_name_unscale)
            imputed_full_segment_unscaled_df[col_name_unscale] = scaler_var_unscale.inverse_transform(imputed_full_segment_scaled[:, col_idx_unscale].reshape(-1, 1)).flatten()
        else:
            imputed_full_segment_unscaled_df[col_name_unscale] = imputed_full_segment_scaled[:, df_with_nans_city_scaled_zeros.columns.get_loc(col_name_unscale)]
    df_final_imputed_unscaled = df_with_nans_city_scaled_zeros.copy()
    for col_orig_unscale in df_with_nans_city_scaled_zeros.columns:
        scaler_orig_var = scaler_dict_vars_city.get(col_orig_unscale)
        if scaler_orig_var and not (df_final_imputed_unscaled[col_orig_unscale] == 0).all():
            df_final_imputed_unscaled[col_orig_unscale] = scaler_orig_var.inverse_transform(df_final_imputed_unscaled[[col_orig_unscale]].values).flatten()
    for col_final_fill in df_final_imputed_unscaled.columns:
        if col_final_fill in imputed_full_segment_unscaled_df.columns:
            col_nan_mask_original = original_nan_mask_df_bool[col_final_fill].values
            df_final_imputed_unscaled.loc[col_nan_mask_original, col_final_fill] = imputed_full_segment_unscaled_df.loc[col_nan_mask_original, col_final_fill]
    return df_final_imputed_unscaled

# --- Función de Métricas ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {}
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan)
            nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results

# --- Script Principal ---
all_detailed_metrics_moment_results = []
script_start_time_moment = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")
print(f"Cargando modelo MOMENT pre-entrenado: {MODEL_NAME_MOMENT_PRETRAINED}")
moment_pipeline_global = None
try:
    moment_pipeline_global = MOMENTPipeline.from_pretrained(
        MODEL_NAME_MOMENT_PRETRAINED, model_kwargs={'task_name': 'reconstruction'}
    )
    moment_pipeline_global.init()
    moment_pipeline_global = moment_pipeline_global.to(device)
    print("Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.")
except Exception as e_model_load:
    print(f"Error crítico cargando modelo MOMENT: {e_model_load}"); traceback.print_exc(); exit()

df_city_ground_truth_moment = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == TARGET_CITY_MOMENT].copy()
if df_city_ground_truth_moment.empty: print(f"No hay datos GT para {TARGET_CITY_MOMENT}."); exit()
for col_gt_fill in ALL_TARGET_COLS_MOMENT: 
    if col_gt_fill in df_city_ground_truth_moment.columns: df_city_ground_truth_moment[col_gt_fill] = df_city_ground_truth_moment[col_gt_fill].ffill().bfill()
if df_city_ground_truth_moment[ALL_TARGET_COLS_MOMENT].isnull().any().any(): print(f"No se pudo rellenar GT para {TARGET_CITY_MOMENT}."); exit()
df_city_ground_truth_moment_reset = df_city_ground_truth_moment.reset_index(drop=True)

scalers_for_city_moment = {}
city_scalers_dir = os.path.join(SCALERS_MOMENT_OUTPUT_DIR, TARGET_CITY_MOMENT.replace(" ", "_")); os.makedirs(city_scalers_dir, exist_ok=True)
for col_scale in ALL_TARGET_COLS_MOMENT: 
    scaler_var = StandardScaler()
    data_to_fit_scaler = df_city_ground_truth_moment_reset[[col_scale]].dropna().values
    if len(data_to_fit_scaler) > 1:
        scaler_var.fit(data_to_fit_scaler)
        scalers_for_city_moment[col_scale] = scaler_var
        joblib.dump(scaler_var, os.path.join(city_scalers_dir, f'scaler_{col_scale}.joblib'))
    else:
        class DummyScaler:
            def fit(self, X): return self
            def transform(self, X): return np.array(X, dtype=float)
            def inverse_transform(self, X): return np.array(X, dtype=float)
        scalers_for_city_moment[col_scale] = DummyScaler()

city_name_safe_moment = TARGET_CITY_MOMENT.replace(" ", "_")
city_imputed_csv_moment_dir_root = os.path.join(IMPUTED_CSVS_MOMENT_OUTPUT_DIR, city_name_safe_moment); os.makedirs(city_imputed_csv_moment_dir_root, exist_ok=True)

for scenario_moment in NAN_SIMULATION_SCENARIOS_MOMENT:
    scenario_name_safe_moment = scenario_moment['name'].replace("%", "percent").replace(" ", "_")
    print(f"\nProcesando Escenario: {scenario_name_safe_moment}")
    sim_data_scenario_dir_moment = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_MOMENT, city_name_safe_moment, scenario_name_safe_moment)
    imputed_moment_scenario_dir = os.path.join(city_imputed_csv_moment_dir_root, scenario_name_safe_moment); os.makedirs(imputed_moment_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_moment): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_moment}."); continue
    
    for rep_moment in range(1, N_REPETITIONS_MOMENT_EVAL + 1):
        print(f"  Repetición: {rep_moment}")
        sim_files_in_dir_moment = sorted([f for f in os.listdir(sim_data_scenario_dir_moment) if f.startswith(f"{city_name_safe_moment}_{scenario_name_safe_moment}_rep{rep_moment}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_moment: print(f"    Archivo simulado no encontrado para rep {rep_moment}."); continue
        sim_file_path_to_load_moment = os.path.join(sim_data_scenario_dir_moment, sim_files_in_dir_moment[0])
        current_seed_loaded_moment = sim_files_in_dir_moment[0].split('_seed')[-1].split('_with_nans.csv')[0]
        try: df_city_with_nans_loaded_moment = pd.read_csv(sim_file_path_to_load_moment)
        except Exception as e_load_rep_m: print(f"    Error cargando {sim_file_path_to_load_moment}: {e_load_rep_m}."); continue
        df_imputed_all_cols_rep = df_city_with_nans_loaded_moment[ALL_TARGET_COLS_MOMENT].copy()
        for col_to_impute in tqdm(ALL_TARGET_COLS_MOMENT, desc=f"  Imputando variables para Rep {rep_moment}", unit="var"):
            df_to_process_univar = df_city_with_nans_loaded_moment[[col_to_impute]].copy()
            scaler_current_var = scalers_for_city_moment.get(col_to_impute)
            df_scaled_univar = pd.DataFrame(index=df_to_process_univar.index, columns=[col_to_impute])
            if scaler_current_var and not df_to_process_univar.isnull().all().iloc[0]:
                df_scaled_univar[col_to_impute] = scaler_current_var.transform(df_to_process_univar.values)
            else:
                df_scaled_univar[col_to_impute] = df_to_process_univar[col_to_impute]
            original_nan_mask_univar_df = df_to_process_univar.isnull()
            df_scaled_filled_univar = df_scaled_univar.fillna(0.0)
            if moment_pipeline_global is None: continue
            df_imputed_univar_final = impute_multivar_with_moment_pipeline(df_scaled_filled_univar, original_nan_mask_univar_df, moment_pipeline_global, device, scalers_for_city_moment, SEQ_LEN_MODEL_MOMENT, IMPUTATION_STEP_MOMENT, BATCH_SIZE_IMPUTE_MOMENT)
            if df_imputed_univar_final is not None and col_to_impute in df_imputed_univar_final.columns:
                nan_mask_for_update = df_imputed_all_cols_rep[col_to_impute].isnull()
                df_imputed_all_cols_rep.loc[nan_mask_for_update, col_to_impute] = df_imputed_univar_final.loc[nan_mask_for_update, col_to_impute]
        df_imputed_moment_final_rep = df_imputed_all_cols_rep
        if df_imputed_moment_final_rep.isnull().all().all():
            print(f"    Falló la imputación MOMENT para rep {rep_moment} (resultado vacío).")
            continue
        df_imputed_moment_final_to_save = df_city_with_nans_loaded_moment[['Date', 'City']].copy()
        for col_save in ALL_TARGET_COLS_MOMENT:
            if col_save in df_imputed_moment_final_rep.columns:
                df_imputed_moment_final_to_save[col_save] = df_imputed_moment_final_rep[col_save]
            else:
                df_imputed_moment_final_to_save[col_save] = np.nan
        imputed_moment_filename = f"{city_name_safe_moment}_{scenario_name_safe_moment}_rep{rep_moment}_seed{current_seed_loaded_moment}_imputed_momentALL.csv"
        imputed_moment_file_save_path = os.path.join(imputed_moment_scenario_dir, imputed_moment_filename)
        final_cols_moment_save = ['Date', 'City'] + ALL_TARGET_COLS_MOMENT
        df_gt_city_reset_for_tf = df_city_ground_truth_moment_reset
        time_features_to_add = [tf for tf in TIME_FEATURES if tf in df_gt_city_reset_for_tf.columns]
        for tf_col_final_m in time_features_to_add:
            if tf_col_final_m not in df_imputed_moment_final_to_save.columns and tf_col_final_m in df_gt_city_reset_for_tf.columns:
                df_imputed_moment_final_to_save[tf_col_final_m] = df_gt_city_reset_for_tf[tf_col_final_m].values[:len(df_imputed_moment_final_to_save)]
            if tf_col_final_m not in final_cols_moment_save:
                final_cols_moment_save.append(tf_col_final_m)
        final_cols_moment_save = [c for c in final_cols_moment_save if c in df_imputed_moment_final_to_save.columns]
        df_imputed_moment_final_to_save = df_imputed_moment_final_to_save[final_cols_moment_save]
        df_imputed_moment_final_to_save.to_csv(imputed_moment_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT guardado: {imputed_moment_filename}")
        nan_mask_introduced_for_metrics_rep = pd.DataFrame(False, index=df_city_ground_truth_moment_reset.index, columns=ALL_TARGET_COLS_MOMENT)
        for col_m_mask_rep in ALL_TARGET_COLS_MOMENT:
            if col_m_mask_rep in df_city_with_nans_loaded_moment.columns and col_m_mask_rep in df_city_ground_truth_moment_reset.columns:
                nan_mask_introduced_for_metrics_rep[col_m_mask_rep] = df_city_with_nans_loaded_moment[col_m_mask_rep].isnull() & ~df_city_ground_truth_moment_reset[col_m_mask_rep].isnull()
        metrics_scores_moment_rep = calculate_metrics_moment(df_city_ground_truth_moment_reset, df_imputed_moment_final_to_save, nan_mask_introduced_for_metrics_rep, ALL_TARGET_COLS_MOMENT)
        for var_name_metric_m in ALL_TARGET_COLS_MOMENT:
            all_detailed_metrics_moment_results.append({'City': TARGET_CITY_MOMENT, 'Scenario': scenario_name_safe_moment, 'Repetition': rep_moment, 'Seed': current_seed_loaded_moment, 'Imputation_Method': MOMENT_METHOD_NAME, 'Variable': var_name_metric_m, 'MAE': metrics_scores_moment_rep[var_name_metric_m]['MAE'], 'RMSE': metrics_scores_moment_rep[var_name_metric_m]['RMSE'], 'NRMAE_range': metrics_scores_moment_rep[var_name_metric_m]['NRMAE_range'], 'NRMSE_range': metrics_scores_moment_rep[var_name_metric_m]['NRMSE_range']})
        print(f"    Métricas calculadas para MOMENT, rep {rep_moment}")
    print(f"  Escenario {scenario_name_safe_moment} completado.")

# --- Guardado de Métricas ---
if all_detailed_metrics_moment_results:
    df_all_detailed_metrics_moment = pd.DataFrame(all_detailed_metrics_moment_results)
    print("\nGenerando resúmenes de métricas MOMENT...")
    sufijo_test_m = "" 
    if len(ALL_TARGET_COLS_MOMENT) == 1:
        sufijo_test_m = f"_SingleVar_{ALL_TARGET_COLS_MOMENT[0]}"
    
    detailed_metrics_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
    df_all_detailed_metrics_moment.to_csv(detailed_metrics_path_m, index=False)
    print(f"  Métricas detalladas MOMENT guardadas en: {detailed_metrics_path_m}")
    if not df_all_detailed_metrics_moment.empty:
        print(df_all_detailed_metrics_moment.head())
    
    grouped_by_var_m = df_all_detailed_metrics_moment.groupby(['Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_var_m.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    
    variables_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
    
    grouped_by_var_m.to_csv(variables_summary_path_m, index=False)
    print(f"  Resumen MOMENT de variables guardado: {variables_summary_path_m}")
    print(grouped_by_var_m)
    
    if len(ALL_TARGET_COLS_MOMENT) > 1:
        CONTAMINANT_COLS_MOMENT_ORIG, AQI_COL_MOMENT_ORIG = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10'], ['AQI']
        df_contaminants_only_m = df_all_detailed_metrics_moment[df_all_detailed_metrics_moment['Variable'].isin(CONTAMINANT_COLS_MOMENT_ORIG)]
        if not df_contaminants_only_m.empty:
            contaminants_combined_summary_m = df_contaminants_only_m.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            contaminants_combined_summary_m.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
            contaminants_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_contaminants_combined_summary_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
            contaminants_combined_summary_m.to_csv(contaminants_summary_path_m, index=False)
            print(f"  Resumen MOMENT contaminantes guardado: {contaminants_summary_path_m}")
            print(contaminants_combined_summary_m)
        
        df_aqi_only_m = df_all_detailed_metrics_moment[df_all_detailed_metrics_moment['Variable'] == AQI_COL_MOMENT_ORIG[0]]
        if not df_aqi_only_m.empty:
            aqi_combined_summary_m = df_aqi_only_m.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
            aqi_combined_summary_m.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
            aqi_summary_path_m = os.path.join(METRICS_MOMENT_OUTPUT_DIR, f'metrics_aqi_combined_summary_{MOMENT_METHOD_NAME}_{TARGET_CITY_MOMENT.replace(" ","_")}{sufijo_test_m}.csv')
            aqi_combined_summary_m.to_csv(aqi_summary_path_m, index=False)
            print(f"  Resumen MOMENT AQI guardado: {aqi_summary_path_m}")
            print(aqi_combined_summary_m)
else:
    print("No se generaron métricas detalladas para MOMENT.")

script_end_time_moment = time.time()
print(f"\nProceso de Imputación con MOMENT Preentrenado completado. Tiempo total: {(script_end_time_moment - script_start_time_moment) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Usando dispositivo: cpu
Cargando modelo MOMENT pre-entrenado: AutonLab/MOMENT-1-large
Modelo MOMENT pre-entrenado cargado, INICIALIZADO y movido al dispositivo.

Procesando Escenario: RandomRows_10percent
  Repetición: 1


  Imputando variables para Rep 1: 100%|██████████| 7/7 [00:25<00:00,  3.58s/var]


    Archivo imputado por MOMENT guardado: Brasilia_RandomRows_10percent_rep1_seed2377262672_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 1
  Repetición: 2


  Imputando variables para Rep 2: 100%|██████████| 7/7 [00:19<00:00,  2.81s/var]


    Archivo imputado por MOMENT guardado: Brasilia_RandomRows_10percent_rep2_seed2500708800_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 2
  Repetición: 3


...

    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_24h_rep3_seed473312285_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 3
  Repetición: 4


  Imputando variables para Rep 4: 100%|██████████| 7/7 [00:20<00:00,  2.94s/var]


    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_24h_rep4_seed587731976_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 4
  Repetición: 5


  Imputando variables para Rep 5: 100%|██████████| 7/7 [00:22<00:00,  3.19s/var]

    Archivo imputado por MOMENT guardado: Brasilia_BlockRows_24h_rep5_seed704623400_imputed_momentALL.csv
    Métricas calculadas para MOMENT, rep 5
  Escenario BlockRows_24h completado.

Generando resúmenes de métricas MOMENT...
  Métricas detalladas MOMENT guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_PRE_Fix7_FinalCorrected\metricas\ALL_metrics_detailed_MOMENT_PRE_Fix7_FinalCorrected_Brasilia.csv
       City              Scenario  Repetition        Seed  \
0  Brasilia  RandomRows_10percent           1  2377262672   
1  Brasilia  RandomRows_10percent           1  2377262672   
2  Brasilia  RandomRows_10percent           1  2377262672   
3  Brasilia  RandomRows_10percent           1  2377262672   
4  Brasilia  RandomRows_10percent           1  2377262672   

                Imputation_Method Variable        MAE       RMSE  NRMAE_range  \
0  MOMENT_PRE_Fix7_FinalCorrected       CO  53.365806  81.704448     0.060369   
1  MOMENT_PRE_Fix7

MOMENT_FT

In [38]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import joblib
import traceback
import time
import shutil
import tempfile
import random
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos Originales (Ground Truth) ---
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load_orig):
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_ground_truth = pd.read_csv(file_path_load_orig)
    print(f"Datos originales (ground truth) cargados desde: {file_path_load_orig}")
except FileNotFoundError:
    print("Error: Air_Quality.csv (original) no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia'] 
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        num_rows_example = 550 
        for i in range(num_rows_example): 
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_ground_truth = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_ground_truth['Date'] = pd.to_datetime(df_air_quality_ground_truth['Date'])
if 'CO2' in df_air_quality_ground_truth.columns:
    df_air_quality_ground_truth = df_air_quality_ground_truth.drop(columns=['CO2'])
df_air_quality_ground_truth['Hour'] = df_air_quality_ground_truth['Date'].dt.hour
df_air_quality_ground_truth['DayOfWeek'] = df_air_quality_ground_truth['Date'].dt.dayofweek
df_air_quality_ground_truth['Month'] = df_air_quality_ground_truth['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# --- Parámetros Configurables para Fine-Tuning ---
TARGET_CITY_FT = "Brasilia"
CONTAMINANT_COLS_FT = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL_FT = ['AQI']
ALL_TARGET_COLS_FT = CONTAMINANT_COLS_FT + AQI_COL_FT

SEQ_LEN_MODEL_FT = 512
IMPUTATION_STEP_FT = 512
BATCH_SIZE_IMPUTE_FT = 16
MODEL_NAME_PRETRAINED_FT = "AutonLab/MOMENT-1-large"

FINETUNE_EPOCHS = 20
FINETUNE_STEP = 512
BATCH_SIZE_FINETUNE = 8
LEARNING_RATE_FINETUNE = 1e-4
PATIENCE_EARLY_STOPPING = 5
MIN_DELTA_EARLY_STOPPING = 1e-5

N_REPETITIONS_FT_EVAL = 5
NAN_SIMULATION_SCENARIOS_MOMENT = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

PROJECT_ROOT_DIR_FT = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR_FT = os.path.join(PROJECT_ROOT_DIR_FT, 'CSVs', 'simulados')
MOMENT_METHOD_NAME_FT = 'MOMENT_FT_v2_Corrected'
IMPUTATIONS_FT_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_FT, 'imputaciones', MOMENT_METHOD_NAME_FT)
METRICS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'metricas')
IMPUTED_CSVS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'CSVs')
MODELS_FT_OUTPUT_DIR = os.path.join(IMPUTATIONS_FT_BASE_DIR, 'modelos')
os.makedirs(METRICS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_FT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_FT_OUTPUT_DIR, exist_ok=True)

# --- Dataset Personalizado ---
class MomentImputationDataset(Dataset):
    def __init__(self, data_array_2d, mask_2d, seq_len, step):
        self.data_array = data_array_2d; self.mask = mask_2d
        self.seq_len = seq_len; self.step = step
        if len(self.data_array) < self.seq_len: self.n_samples, self.indices = 1, [0]
        elif len(self.data_array) == 0: self.n_samples, self.indices = 0, []
        else:
            self.indices = list(range(0, len(self.data_array) - self.seq_len + 1, self.step))
            last_possible_start = len(self.data_array) - self.seq_len
            if not self.indices or self.indices[-1] < last_possible_start:
                if not self.indices or last_possible_start > self.indices[-1]: self.indices.append(last_possible_start)
            self.n_samples = len(self.indices)
            if self.n_samples == 0 and len(self.data_array) >= self.seq_len: self.indices, self.n_samples = [0], 1
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        start = self.indices[idx]
        seq_data = self.data_array[start:start + self.seq_len]
        seq_mask = self.mask[start:start + self.seq_len]
        if len(seq_data) < self.seq_len:
            pad = self.seq_len - len(seq_data)
            seq_data = np.pad(seq_data, ((0, pad), (0, 0)), 'constant', constant_values=0.0)
            seq_mask = np.pad(seq_mask, ((0, pad), (0, 0)), 'constant', constant_values=False)
        timeseries_out = torch.tensor(seq_data, dtype=torch.float32).transpose(0, 1)
        input_mask_out = torch.tensor(seq_mask, dtype=torch.bool).squeeze()
        return {'timeseries': timeseries_out, 'input_mask': input_mask_out}

# --- Función de Entrenamiento (Fine-Tuning) ---
def train_moment_model(model, train_loader, device, epochs, variable_name, model_save_path):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE_FINETUNE)
    criterion = nn.MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=PATIENCE_EARLY_STOPPING // 2, verbose=False)
    
    best_loss = float('inf')
    epochs_no_improve = 0
    print(f"Iniciando fine-tuning para '{variable_name}' ({epochs} épocas máx)...")
    
    for epoch in range(epochs):
        total_epoch_loss = 0
        for batch_data in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            timeseries = batch_data['timeseries'].to(device)
            input_mask = batch_data['input_mask'].to(device) 
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=timeseries, input_mask=input_mask)
                loss = criterion(outputs.reconstruction, timeseries)
            except Exception as e: continue
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_epoch_loss += loss.item()
            
        avg_epoch_loss = total_epoch_loss / len(train_loader) if len(train_loader) > 0 else 0
        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1} Loss ({variable_name}): {avg_epoch_loss:.6f}")
        scheduler.step(avg_epoch_loss)
        
        if avg_epoch_loss < best_loss - MIN_DELTA_EARLY_STOPPING:
            best_loss = avg_epoch_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= PATIENCE_EARLY_STOPPING:
            print(f"  Early stopping en epoch {epoch+1}.")
            break
            
    print(f"Fine-tuning para '{variable_name}' completado. Mejor pérdida: {best_loss:.6f}")
    if os.path.exists(model_save_path):
        model.load_state_dict(torch.load(model_save_path, map_location=device))
        print(f"  Mejor modelo para '{variable_name}' cargado desde {model_save_path}")
    return model

# --- Función de Imputación (CORREGIDA) ---
def impute_with_finetuned_model(df_with_nans_scaled_zeros, original_nan_mask_df, model_ft, device, scaler, seq_len, step, batch_size):
    model_ft.eval()
    imputation_dataset = MomentImputationDataset(df_with_nans_scaled_zeros.values, ~original_nan_mask_df.values, seq_len, step)
    if len(imputation_dataset) == 0: return None
    imputation_loader = DataLoader(imputation_dataset, batch_size=batch_size, shuffle=False)
    
    n_rows, n_features = df_with_nans_scaled_zeros.shape
    accumulated_preds = np.zeros((n_rows, n_features)); accumulated_counts = np.zeros((n_rows, n_features))

    with torch.no_grad():
        for batch_idx, batch_data in enumerate(imputation_loader):
            timeseries = batch_data['timeseries'].to(device); input_mask = batch_data['input_mask'].to(device)
            try: outputs = model_ft(x_enc=timeseries, input_mask=input_mask)
            except Exception as e: continue
            preds_batch = outputs.reconstruction.cpu().numpy()
            
            for i in range(preds_batch.shape[0]):
                global_sample_idx = batch_idx * batch_size + i
                if global_sample_idx >= len(imputation_loader.dataset.indices): continue
                start = imputation_loader.dataset.indices[global_sample_idx]
                end = min(start + seq_len, n_rows)
                window_len = end - start
                
                if window_len > 0:
                    pred_win = preds_batch[i, :, :window_len].T
                    accumulated_preds[start:end] += pred_win
                    accumulated_counts[start:end] += 1
    
    accumulated_counts[accumulated_counts == 0] = 1
    imputed_scaled = accumulated_preds / accumulated_counts
    imputed_unscaled = scaler.inverse_transform(imputed_scaled)
    
    # Iniciar con los valores originales (des-escalados)
    df_imputed_final = pd.DataFrame(scaler.inverse_transform(df_with_nans_scaled_zeros), columns=df_with_nans_scaled_zeros.columns, index=df_with_nans_scaled_zeros.index)
    imputed_unscaled_df = pd.DataFrame(imputed_unscaled, columns=df_imputed_final.columns, index=df_imputed_final.index)
    
    # Actualizar solo donde había NaNs
    df_imputed_final = df_imputed_final.where(~original_nan_mask_df, imputed_unscaled_df)
    return df_imputed_final

# --- Función de Métricas ---
def calculate_metrics_moment(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset, cols_to_eval):
    results = {};
    for col in cols_to_eval:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns: continue
        true_col_vals, pred_col_vals, mask_col_flags = y_true_df_reset[col].values, y_pred_df_reset[col].values, nan_mask_for_metrics_df_reset[col].values
        true_at_nans, pred_at_nans = true_col_vals[mask_col_flags], pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans); true_at_nans_valid, pred_at_nans_valid = true_at_nans[valid_pred_mask], pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid); rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            original_col_full_series = y_true_df_reset[col].dropna(); original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0
            nrmae_range = (mae / original_col_range if original_col_range > 1e-9 else np.nan); nrmse_range = (rmse / original_col_range if original_col_range > 1e-9 else np.nan)
            results[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': nrmae_range, 'NRMSE_range': nrmse_range})
        except Exception: pass
    return results
    
# --- Script Principal ---
all_detailed_metrics_ft_results = []
script_start_time_ft = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"; print(f"Usando dispositivo: {device}")

df_city_ground_truth_ft = df_air_quality_ground_truth[df_air_quality_ground_truth['City'] == TARGET_CITY_FT].copy()
if df_city_ground_truth_ft.empty: print(f"No hay datos GT para {TARGET_CITY_FT}."); exit()
for col_gt_fill in ALL_TARGET_COLS_FT: 
    if col_gt_fill in df_city_ground_truth_ft.columns: df_city_ground_truth_ft[col_gt_fill] = df_city_ground_truth_ft[col_gt_fill].ffill().bfill()
if df_city_ground_truth_ft[ALL_TARGET_COLS_FT].isnull().any().any(): print(f"No se pudo rellenar GT para {TARGET_CITY_FT}."); exit()
df_city_ground_truth_ft_reset = df_city_ground_truth_ft.reset_index(drop=True)

finetuned_models = {}; variable_scalers = {}
city_models_dir = os.path.join(MODELS_FT_OUTPUT_DIR, TARGET_CITY_FT.replace(" ", "_")); os.makedirs(city_models_dir, exist_ok=True)

for var_to_process in ALL_TARGET_COLS_FT:
    print(f"\n--- Preparando Fine-Tuning para la variable: {var_to_process} ---")
    scaler = StandardScaler()
    data_for_scaler = df_city_ground_truth_ft_reset[[var_to_process]].dropna().values
    if len(data_for_scaler) < SEQ_LEN_MODEL_FT: print(f"Datos insuficientes para entrenar {var_to_process}. Saltando."); continue
    scaler.fit(data_for_scaler); variable_scalers[var_to_process] = scaler
    
    train_data_scaled = scaler.transform(data_for_scaler)
    train_mask = np.ones_like(train_data_scaled, dtype=bool) 
    train_dataset = MomentImputationDataset(train_data_scaled, train_mask, SEQ_LEN_MODEL_FT, FINETUNE_STEP)
    if len(train_dataset) == 0: print(f"Dataset de entrenamiento para {var_to_process} vacío. Saltando."); continue
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_FINETUNE, shuffle=True)

    model_ft = MOMENTPipeline.from_pretrained(MODEL_NAME_PRETRAINED_FT, model_kwargs={'task_name': 'reconstruction'})
    model_ft.init(); model_ft = model_ft.to(device)
    model_save_path = os.path.join(city_models_dir, f'moment_ft_{var_to_process}.pth')
    
    finetuned_model = train_moment_model(model_ft, train_loader, device, FINETUNE_EPOCHS, var_to_process, model_save_path)
    finetuned_models[var_to_process] = finetuned_model

print("\n--- Fine-Tuning completado para todas las variables ---")

city_name_safe_ft = TARGET_CITY_FT.replace(" ", "_")
city_imputed_csv_ft_dir_root = os.path.join(IMPUTED_CSVS_FT_OUTPUT_DIR, city_name_safe_ft); os.makedirs(city_imputed_csv_ft_dir_root, exist_ok=True)

for scenario_ft in NAN_SIMULATION_SCENARIOS_FT:
    scenario_name_safe_ft = scenario_ft['name'].replace("%", "percent").replace(" ", "_")
    print(f"\nProcesando Escenario: {scenario_name_safe_ft}")
    sim_data_scenario_dir_ft = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR_FT, city_name_safe_ft, scenario_name_safe_ft)
    imputed_ft_scenario_dir = os.path.join(city_imputed_csv_ft_dir_root, scenario_name_safe_ft); os.makedirs(imputed_ft_scenario_dir, exist_ok=True)
    if not os.path.exists(sim_data_scenario_dir_ft): print(f"    Dir simulado no encontrado: {sim_data_scenario_dir_ft}."); continue

    for rep_ft in range(1, N_REPETITIONS_FT_EVAL + 1):
        print(f"  Repetición: {rep_ft}")
        sim_files_in_dir_ft = sorted([f for f in os.listdir(sim_data_scenario_dir_ft) if f.startswith(f"{city_name_safe_ft}_{scenario_name_safe_ft}_rep{rep_ft}") and f.endswith("_with_nans.csv")])
        if not sim_files_in_dir_ft: print(f"    Archivo simulado no encontrado para rep {rep_ft}."); continue
        sim_file_path_to_load_ft = os.path.join(sim_data_scenario_dir_ft, sim_files_in_dir_ft[0])
        current_seed_loaded_ft = sim_files_in_dir_ft[0].split('_seed')[-1].split('_with_nans.csv')[0]
        try: df_city_with_nans_loaded_ft = pd.read_csv(sim_file_path_to_load_ft)
        except Exception as e_load_rep_ft: print(f"    Error cargando {sim_file_path_to_load_ft}: {e_load_rep_ft}."); continue

        df_imputed_all_cols_rep_ft = df_city_with_nans_loaded_ft.copy()

        for var_to_impute in tqdm(ALL_TARGET_COLS_FT, desc=f"  Imputando variables para Rep {rep_ft}", unit="var"):
            if var_to_impute not in finetuned_models: continue
            
            model_to_use = finetuned_models[var_to_impute]; scaler_to_use = variable_scalers[var_to_impute]

            df_univar_with_nans = df_city_with_nans_loaded_ft[[var_to_impute]]
            original_nan_mask_univar = df_univar_with_nans.isnull()
            df_univar_scaled_zeros = pd.DataFrame(scaler_to_use.transform(df_univar_with_nans.fillna(0.0)), columns=[var_to_impute], index=df_univar_with_nans.index)

            df_imputed_univar_final = impute_with_finetuned_model(
                df_univar_scaled_zeros, original_nan_mask_univar,
                model_to_use, device, scaler_to_use,
                SEQ_LEN_MODEL_FT, IMPUTATION_STEP_FT, BATCH_SIZE_IMPUTE_FT
            )
            
            if df_imputed_univar_final is not None:
                df_imputed_all_cols_rep_ft[var_to_impute] = df_imputed_univar_final[var_to_impute]

        imputed_ft_filename = f"{city_name_safe_ft}_{scenario_name_safe_ft}_rep{rep_ft}_seed{current_seed_loaded_ft}_imputed_momentFT.csv"
        imputed_ft_file_save_path = os.path.join(imputed_ft_scenario_dir, imputed_ft_filename)
        final_cols_ft_save = ['Date', 'City'] + ALL_TARGET_COLS_FT
        for tf_col_final in TIME_FEATURES:
            if tf_col_final not in df_imputed_all_cols_rep_ft.columns and tf_col_final in df_city_ground_truth_ft_reset.columns:
                df_imputed_all_cols_rep_ft[tf_col_final] = df_city_ground_truth_ft_reset[tf_col_final].values[:len(df_imputed_all_cols_rep_ft)]
            if tf_col_final not in final_cols_ft_save: final_cols_ft_save.append(tf_col_final)
        df_imputed_all_cols_rep_ft = df_imputed_all_cols_rep_ft[[c for c in final_cols_ft_save if c in df_imputed_all_cols_rep_ft.columns]]
        df_imputed_all_cols_rep_ft.to_csv(imputed_ft_file_save_path, index=False)
        print(f"    Archivo imputado por MOMENT-FT guardado: {imputed_ft_filename}")
        
        nan_mask_introduced_for_metrics_rep_ft = pd.DataFrame(False, index=df_city_ground_truth_ft_reset.index, columns=ALL_TARGET_COLS_FT)
        for col_m_mask_rep in ALL_TARGET_COLS_FT:
            if col_m_mask_rep in df_city_with_nans_loaded_ft.columns and col_m_mask_rep in df_city_ground_truth_ft_reset.columns:
                nan_mask_introduced_for_metrics_rep_ft[col_m_mask_rep] = df_city_with_nans_loaded_ft[col_m_mask_rep].isnull() & ~df_city_ground_truth_ft_reset[col_m_mask_rep].isnull()
        
        metrics_scores_ft_rep = calculate_metrics_moment(df_city_ground_truth_ft_reset, df_imputed_all_cols_rep_ft, nan_mask_introduced_for_metrics_rep_ft, ALL_TARGET_COLS_FT)
        
        for var_name_metric in ALL_TARGET_COLS_FT:
            all_detailed_metrics_ft_results.append({'City': TARGET_CITY_FT, 'Scenario': scenario_name_safe_ft, 'Repetition': rep_ft, 'Seed': current_seed_loaded_ft, 'Imputation_Method': MOMENT_METHOD_NAME_FT, 'Variable': var_name_metric, 'MAE': metrics_scores_ft_rep[var_name_metric]['MAE'], 'RMSE': metrics_scores_ft_rep[var_name_metric]['RMSE'], 'NRMAE_range': metrics_scores_ft_rep[var_name_metric]['NRMAE_range'], 'NRMSE_range': metrics_scores_ft_rep[var_name_metric]['NRMSE_range']})
        print(f"    Métricas calculadas para MOMENT-FT, rep {rep_ft}")
    print(f"  Escenario {scenario_name_safe_ft} completado.")

if all_detailed_metrics_ft_results:
    df_all_detailed_metrics_ft = pd.DataFrame(all_detailed_metrics_ft_results)
    print("\nGenerando resúmenes de métricas MOMENT-FT...")
    detailed_metrics_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'ALL_metrics_detailed_{MOMENT_METHOD_NAME_FT}_{TARGET_CITY_FT.replace(" ","_")}.csv')
    df_all_detailed_metrics_ft.to_csv(detailed_metrics_path_ft, index=False)
    print(f"  Métricas detalladas MOMENT-FT guardadas en: {detailed_metrics_path_ft}")
    if not df_all_detailed_metrics_ft.empty: print(df_all_detailed_metrics_ft.head())
    grouped_by_var_ft = df_all_detailed_metrics_ft.groupby(['Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_var_ft.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    variables_summary_path_ft = os.path.join(METRICS_FT_OUTPUT_DIR, f'metrics_variables_summary_{MOMENT_METHOD_NAME_FT}_{TARGET_CITY_FT.replace(" ","_")}.csv')
    grouped_by_var_ft.to_csv(variables_summary_path_ft, index=False)
    print(f"  Resumen MOMENT-FT de variables guardado: {variables_summary_path_ft}"); print(grouped_by_var_ft)
else:
    print("No se generaron métricas detalladas para MOMENT-FT.")

script_end_time_ft = time.time()
print(f"\nProceso de Fine-Tuning e Imputación con MOMENT completado. Tiempo total: {(script_end_time_ft - script_start_time_ft) / 60:.2f} minutos.")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
Usando dispositivo: cpu

--- Preparando Fine-Tuning para la variable: CO ---
Iniciando fine-tuning para 'CO' (20 épocas máx)...


  Epoch 1 Loss (CO): 0.187409


  Epoch 5 Loss (CO): 0.190900


  Early stopping en epoch 6.
Fine-tuning para 'CO' completado. Mejor pérdida: 0.187409
  Mejor modelo para 'CO' cargado desde c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_v2_Corrected\modelos\Brasilia\moment_ft_CO.pth

--- Preparando Fine-Tuning para la variable: NO2 ---
Iniciando fine-tuning para 'NO2' (20 épocas máx)...


  Epoch 1 Loss (NO2): 0.319694


...

    Archivo imputado por MOMENT-FT guardado: Brasilia_BlockRows_8h_rep3_seed2255836414_imputed_momentFT.csv
    Métricas calculadas para MOMENT-FT, rep 3
  Repetición: 4


  Imputando variables para Rep 4: 100%|██████████| 7/7 [00:23<00:00,  3.29s/var]


    Archivo imputado por MOMENT-FT guardado: Brasilia_BlockRows_8h_rep4_seed2385710956_imputed_momentFT.csv
    Métricas calculadas para MOMENT-FT, rep 4
  Repetición: 5


  Imputando variables para Rep 5: 100%|██████████| 7/7 [00:21<00:00,  3.12s/var]

    Archivo imputado por MOMENT-FT guardado: Brasilia_BlockRows_8h_rep5_seed2503372948_imputed_momentFT.csv
    Métricas calculadas para MOMENT-FT, rep 5
  Escenario BlockRows_8h completado.

Generando resúmenes de métricas MOMENT-FT...
  Métricas detalladas MOMENT-FT guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\imputaciones\MOMENT_FT_v2_Corrected\metricas\ALL_metrics_detailed_MOMENT_FT_v2_Corrected_Brasilia.csv
       City              Scenario  Repetition        Seed  \
0  Brasilia  RandomRows_10percent           1  2377262672   
1  Brasilia  RandomRows_10percent           1  2377262672   
2  Brasilia  RandomRows_10percent           1  2377262672   
3  Brasilia  RandomRows_10percent           1  2377262672   
4  Brasilia  RandomRows_10percent           1  2377262672   

        Imputation_Method Variable        MAE       RMSE  NRMAE_range  \
0  MOMENT_FT_v2_Corrected       CO  46.566176  73.870741     0.052677   
1  MOMENT_FT_v2_Corrected      NO2   4.6